In [ ]:
""" Configuration cell — thesis figures (see paper_plots.ipynb for the paper ones)"""

from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json

import morethemes as mt
mt.set_theme("minimal")

VERIF_DIR = Path("data/verification")
CONV_DIR = Path("data/discretisation_convergence")
FIG_DIR = Path("figures")
THESIS_FIG_DIR = Path("../masters_thesis/fig")   # figures are written straight into the thesis
FIG_DIR.mkdir(exist_ok=True, parents=True)

HOURS_TO_SEC = 3600
SEC_TO_HOURS = 1 / HOURS_TO_SEC
M_TO_CM = 1e2

# matplotlib stamps every PDF with a creation date, so a rerun rewrites each file even
# when nothing was redrawn. Pinning SOURCE_DATE_EPOCH makes the output reproducible, so
# the publish step below can tell a real change from a fresh timestamp.
import os

os.environ["SOURCE_DATE_EPOCH"] = "0"

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["STIXGeneral", "Times New Roman", "Times", "DejaVu Serif"],
    "mathtext.fontset": "stix",
    "font.size": 9,
    "axes.labelsize": 9,
    "axes.titlesize": 9,
    "legend.fontsize": 8,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "axes.linewidth": 0.7,
    "lines.linewidth": 1.3,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.top": False,
    "ytick.right": False,
    "xtick.minor.visible": True,
    "ytick.minor.visible": True,
    "xtick.major.size": 3.5,
    "ytick.major.size": 3.5,
    "xtick.minor.size": 2,
    "ytick.minor.size": 2,
    "xtick.major.width": 0.7,
    "ytick.major.width": 0.7,
    "xtick.minor.width": 0.5,
    "ytick.minor.width": 0.5,
    "legend.frameon": False,
    "savefig.bbox": "tight",
    "savefig.dpi": 300,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

In [ ]:
# %% ===================== CONTROLS : postprocessing thresholds =====================
""" The only knobs of the analysis. They are NOT run conditions: nothing here was baked into the
    CSVs, so changing a value and re-running the figure/summary cells below is enough -- no need
    to re-run any study. """

# a decay whose exponential fit leaves a normalized RMSE above this is called non-exponential:
# tau still has a value, but it stops being the whole story (red edge on every scatter).
RMSE_THRESHOLD = 1e-4

# a sample is "clean" for a given figure when the two governing groups that are NOT on its x axis
# are both below their own threshold -- so the trend against x is not polluted by the other two.
# One threshold per group, set them independently.
GROUP_THRESHOLDS = {
    "Pi": 0.1,
    "G_mix": 0.1,
    "G_P": 0.1,
}

GROUP_COL = {"Pi": "Pi", "G_mix": "G_mix", "G_P": "G_P"}          # column holding each group
GROUP_LABEL = {"Pi": r"\langle\Pi\rangle", "G_mix": r"\Gamma_\mathrm{mix}",
               "G_P": r"\Gamma_P"}


def flagged_mask(df):
    """Samples whose decay is not a single exponential, at the current RMSE_THRESHOLD."""
    return (df.fit_rmse_norm > RMSE_THRESHOLD).to_numpy()


def clean_mask(df, x_group):
    """Samples for which the two groups other than `x_group` are below their own thresholds."""
    others = [g for g in GROUP_THRESHOLDS if g != x_group]
    return np.logical_and.reduce(
        [(df[GROUP_COL[g]] < GROUP_THRESHOLDS[g]).to_numpy() for g in others]
    )


def others_label(x_group):
    """Legend text spelling out the condition applied to the two other groups."""
    return ", ".join(rf"${GROUP_LABEL[g]} < {GROUP_THRESHOLDS[g]:g}$"
                     for g in GROUP_THRESHOLDS if g != x_group)


print("thresholds in use -> RMSE:", RMSE_THRESHOLD, "| groups:", GROUP_THRESHOLDS)

In [ ]:
# %% ===================== HELPERS =====================
def _read_units(path):
    """Read the leading '# units: <unit>' comment line, if present."""
    with open(path) as f:
        first = f.readline().strip()
    m = re.match(r"#\s*units:\s*(.*)", first)
    return m.group(1) if m else ""


def load_export(path: Path):
    """Load an export, inferring its kind from the CSV column layout.
        profile : col 0 'x_metres', then one column per time step 't=<s>s'
        static  : columns 'x_metres', 'value'
        series  : columns 't_seconds', 'value'
    """
    unit = _read_units(path)
    df = pd.read_csv(path, comment="#")
    cols = list(df.columns)
    if cols[0] == "x_metres" and all(c.startswith("t=") for c in cols[1:]):
        times = np.array([float(re.search(r"t=([\d.eE+-]+)s", c).group(1)) for c in cols[1:]])
        return dict(kind="profile", unit=unit, x=df["x_metres"].to_numpy(),
                    times=times, data=df[cols[1:]].to_numpy().T)
    elif cols == ["x_metres", "value"]:
        return dict(kind="static", unit=unit, x=df["x_metres"].to_numpy(),
                    data=df["value"].to_numpy())
    elif cols == ["t_seconds", "value"]:
        return dict(kind="series", unit=unit, times=df["t_seconds"].to_numpy(),
                    data=df["value"].to_numpy())
    raise ValueError(f"Unrecognized export layout in {path}: columns={cols}")


def select_nearest(times, targets):
    """For each target time, return the index of the closest available time."""
    return [int(np.argmin(np.abs(times - t))) for t in targets]


def richardson(f, r):
    """Observed order p, extrapolate and GCI from the finest 3 of a coarse->fine
    series (NaN when the triplet is non-monotone). Roache/Celik GCI convention:
    f1 = finest grid, f2 = next-coarsest, f3 = coarsest of the three."""
    f3, f2, f1 = f[-3:]
    eps21, eps32 = f2 - f1, f3 - f2
    if eps32 == 0 or eps21 == 0 or np.sign(eps21) != np.sign(eps32):
        return np.nan, f1, np.nan
    p = np.log(abs(eps32) / abs(eps21)) / np.log(r)
    f_exact = f1 + (f1 - f2) / (r**p - 1)
    gci = 1.25 * abs((f1 - f2) / f1) / (r**p - 1)
    return p, f_exact, gci


def ref_slope(ax, xdata, slope, xref, eref, label, color="0.45"):
    """Dashed reference guide of order `slope` through (xref, eref).

    The label sits in the top right corner rather than on the guide: the data runs corner to
    corner on these axes, so anything placed along the line collides with it.
    """
    x = np.array([xdata.min(), xdata.max()])
    y = eref * (x / xref) ** slope
    ax.plot(x, y, ls="--", lw=0.9, color=color, zorder=0)
    ax.text(0.97, 0.95, label, transform=ax.transAxes, color=color, fontsize=7,
            va="top", ha="right")

In [ ]:
# %% ===================== CELL V0 : load the verification case =====================
""" Verification against the analytical solution — data from
    generate_data.verification_case() """
import sys
sys.path.insert(0, ".")
from generate_data import _make_scaled_input
from sparging import get_sim_input_LIBRA_Pi, ureg
from scipy.integrate import quad

VARIANT_LABEL = {"spp": r"SPP, $\langle\Pi\rangle = %.3g$", "ppl": r"PPL, $\langle\Pi\rangle = %.3g$"}
PANEL = {"spp": "(a)", "ppl": "(b)"}
SIM_KW = dict(lw=2.6, alpha=0.45, solid_capstyle="round")          # thick, faded: the model
ANA_KW = dict(color="0.15", lw=1.0, ls=(0, (3, 2)), zorder=5)      # thin, crisp: the analytical
VARIANT_COLOR = {"spp": "#4477AA", "ppl": "#EE7733"}

VERIF = {}
for variant in ("spp", "ppl"):
    d = VERIF_DIR / variant
    ard = pd.read_csv(d / "ard_inputs.csv").set_index("key")
    meta = json.load(open(d / "metadata.json"))
    inp = _make_scaled_input(
        get_sim_input_LIBRA_Pi().height * meta["height_scale"], "K_s", meta["k_s_scale"]
    )
    VERIF[variant] = dict(
        ard=ard, meta=meta, inp=inp,
        tau_s=inp.get_tau().to("s").magnitude,
        Pi=float(ard.loc["Pi_ave", "value"]),
        H=float(ard.loc["H", "value"]),
        A=float(ard.loc["A", "value"]),
        K_s=float(ard.loc["K_s", "value"]),
        dt_s=float(ard.loc["dt", "value"]),
        P_T2=load_export(d / "P_T2.csv"),
        inventory=load_export(d / "n_T2_salt.csv"),
    )
    print(f"{variant}: Pi={VERIF[variant]['Pi']:.3g}, G_P={float(ard.loc['G_P','value']):.3g}, "
          f"G_mix={float(ard.loc['G_mix','value']):.2g}, tau={VERIF[variant]['tau_s']/3600:.2f} h, "
          f"n_cells={int(ard.loc['n_cells','value'])}, n_steps={int(ard.loc['n_steps','value'])}")


def pi_star(inp, z_m):
    """Cumulated partial pressure number seen by a bubble from the sparger up to z:
    Pi*(z) = (1/H) int_0^z Pi(s) ds, so that Pi*(H) = <Pi>."""
    H = inp.height.to("m").magnitude
    f = lambda s: inp.get_Pi(s * ureg.m).magnitude
    return np.array([quad(f, 0, zi)[0] / H for zi in np.atleast_1d(z_m)])

In [ ]:
# %% ===================== CELL V1 : gas partial pressure profile vs analytical =====================
# analytical: P_T2(z) = <c_T2>^l / K_s * (1 - exp(-Pi*(z))), with <c_T2>^l taken from the
# simulated inventory at the same instant (the liquid is uniform: G_mix ~ 1e-6).
fig, axes = plt.subplots(2, 1, figsize=(3.5, 4.6), sharex=True)

for ax, variant in zip(axes, ("spp", "ppl")):
    v = VERIF[variant]
    e, inv = v["P_T2"], v["inventory"]
    idx = select_nearest(e["times"], np.array([0.2, 1.0, 2.0]) * v["tau_s"])
    colors = plt.cm.viridis(np.linspace(0.05, 0.75, len(idx)))
    zs = np.linspace(0, v["H"], 200)
    saturation = 1 - np.exp(-pi_star(v["inp"], zs))

    err = []
    for c, i in zip(colors, idx):
        c_mean = np.interp(e["times"][i], inv["times"], inv["data"]) / (v["A"] * v["H"])
        ana = c_mean / v["K_s"] * saturation
        ax.plot(e["x"] * M_TO_CM, e["data"][i], color=c, **SIM_KW,
                label=fr"$t = {e['times'][i] / HOURS_TO_SEC:.0f}\,$h")
        ax.plot(zs * M_TO_CM, ana, **ANA_KW)
        err.append(np.max(np.abs(np.interp(e["x"], zs, ana) - e["data"][i])) / np.max(ana))

    ax.set_title(VARIANT_LABEL[variant] % v["Pi"], fontsize=9)
    ax.set_xlim(0, v["H"] * M_TO_CM)
    ax.set_ylabel(r"$P_{T_2}\ \mathrm{[Pa]}$")
    ax.legend(loc="upper left", fontsize=7)
    print(f"{variant}: max |P_T2 - analytical| / max(P_T2) = {max(err):.2e}")

axes[-1].set_xlabel(r"$z\ \mathrm{[cm]}$")
axes[0].plot([], [], color="0.4", **SIM_KW, label="1D ARD model")
axes[0].plot([], [], **ANA_KW, label="analytical")
handles = axes[0].get_legend_handles_labels()
# the curves rise from the origin, so the upper left is the free corner in both panels
axes[1].legend(handles[0][-2:], handles[1][-2:], loc="upper left", fontsize=7)

fig.tight_layout()
fig.savefig(FIG_DIR / "verification_P_T2_profile.pdf")

In [ ]:
# %% ===================== CELL V2 : inventory decay vs analytical =====================
# n_T2(t) = n_T2(0) exp(-t/tau), plotted against absolute time so the two regimes keep their
# own decay rate (normalising by tau would collapse them onto the same curve).
fig, (ax, axr) = plt.subplots(2, 1, figsize=(3.5, 4.6), sharex=True,
                              gridspec_kw={"height_ratios": [2.2, 1]})

for variant in ("spp", "ppl"):
    v = VERIF[variant]
    t, n = v["inventory"]["times"], v["inventory"]["data"]
    ana = n[0] * np.exp(-t / v["tau_s"])
    ana_dt = n[0] * np.exp(-t / (v["tau_s"] + v["dt_s"] / 2))
    color = VARIANT_COLOR[variant]

    ax.plot(t / HOURS_TO_SEC, n, color=color, **SIM_KW,
            label=VARIANT_LABEL[variant] % v["Pi"])
    ax.plot(t / HOURS_TO_SEC, ana, **ANA_KW)
    axr.plot(t / HOURS_TO_SEC, (n - ana) / ana * 100, color=color, lw=1.0, ls="-")
    axr.plot(t / HOURS_TO_SEC, (n - ana_dt) / ana_dt * 100, color=color, lw=1.0, ls="--")
    print(f"{variant}: max deviation vs tau = {np.max(np.abs(n - ana) / ana):.2e}, "
          f"vs tau + dt/2 = {np.max(np.abs(n - ana_dt) / ana_dt):.2e}")

ax.set_yscale("log")
ax.set_ylabel(r"$n_{T_2}\ \mathrm{[mol]}$")
ax.plot([], [], **ANA_KW, label="analytical")
ax.legend(loc="lower left", fontsize=7)
axr.axhline(0, color="0.6", lw=0.6)
axr.set_xlabel(r"$t\ \mathrm{[h]}$")
axr.set_ylabel("residual [%]", fontsize=8)
axr.plot([], [], color="0.3", lw=1.0, ls="-", label=r"vs $\tau$")
axr.plot([], [], color="0.3", lw=1.0, ls="--", label=r"vs $\tau + \Delta t/2$")
axr.legend(loc="lower left", fontsize=6, ncol=2)

fig.tight_layout()
fig.savefig(FIG_DIR / "verification_inventory.pdf")

In [ ]:
# %% ===================== CELL C0 : load the convergence study =====================
""" Mesh / time-step convergence on the nominal LIBRA-Pi input """
_cs = json.load(open(CONV_DIR / "convergence_data.json"))
CS_META = _cs["metadata"]
R_REF = CS_META["refinement_ratio"]
TAU_ANA = CS_META["tau_s"] * SEC_TO_HOURS


def cs_sweep(axis):
    """Return (dt[s], dx[m], tau_fitted[h]) arrays, ordered coarse -> fine."""
    recs = _cs[f"{axis}_sweep"]
    return (np.array([x["dt_s"] for x in recs]),
            np.array([x["dx_m"] for x in recs]),
            np.array([x["tau_fitted_s"] for x in recs]) * SEC_TO_HOURS)


# cell Peclet, u_g dx / E_g, is the criterion that fixes dx (sec:numerical_error). The stored
# Bo of this dataset predates the Bo redefinition, so it is u_g H / E_g, i.e. exactly the
# superficial-velocity form the mesh criterion uses: u_g/E_g = Bo/H, no rerun needed.
PE_PER_DX = CS_META["Bo"] / CS_META["H_m"]

P_T, TAU_INF_T, GCI_T = richardson(cs_sweep("dt")[2], R_REF)
P_X, TAU_INF_X, GCI_X = richardson(cs_sweep("dx")[2], R_REF)
print(f"Pi={CS_META['Pi']:.3f}, Bo={CS_META['Bo']:.1f}, analytical tau={TAU_ANA:.4f} h")
print(f"time  : p={P_T:.3f}, tau_inf={TAU_INF_T:.5f} h, GCI={GCI_T*100:.3f} %")
print(f"space : p={P_X:.3f}, tau_inf={TAU_INF_X:.5f} h, GCI={GCI_X*100:.2e} %")
_dxs = cs_sweep("dx")[1]
print(f"cell Peclet = {PE_PER_DX:.2f} * dx  ->  "
      f"{PE_PER_DX*_dxs.max():.2f} (coarsest) .. {PE_PER_DX*_dxs.min():.3f} (finest)")

In [ ]:
# %% ===================== CELL C1 : log-log refinement =====================
# discretisation error of tau_fitted against the Richardson extrapolate of each sweep
fig, (axt, axx) = plt.subplots(1, 2, figsize=(7.0, 2.9))

dt, _, tau_t = cs_sweep("dt")
_, dx, tau_x = cs_sweep("dx")
e_t = np.abs(tau_t - TAU_INF_T) / TAU_INF_T
e_x = np.abs(tau_x - TAU_INF_X) / TAU_INF_X

axt.loglog(dt / (TAU_ANA * HOURS_TO_SEC), e_t, "o-", color="#4477AA", ms=4, mfc="white")
ref_slope(axt, dt / (TAU_ANA * HOURS_TO_SEC), 1.0, dt[-1] / (TAU_ANA * HOURS_TO_SEC),
          e_t[-1], r"order 1")
axt.set_xlabel(r"$\Delta t / \tau_\mathrm{ana}$")
axt.set_ylabel(r"$|\tau_\mathrm{fit} - \tau_\infty| / \tau_\infty$")
axt.set_title("time step", fontsize=9)

pe = PE_PER_DX * dx        # cell Peclet: the criterion that actually fixes the mesh
axx.loglog(pe, e_x, "s-", color="#228833", ms=4, mfc="white")
ref_slope(axx, pe, 2.0, pe[-1], e_x[-1], r"order 2")
axx.set_xlabel(r"$\mathrm{Pe}_\mathrm{cell} = u_g\,\Delta x / E_g$")
axx.set_title("mesh", fontsize=9)

for ax, lab in zip((axt, axx), ("(a)", "(b)")):
    ax.text(0.03, 0.95, lab, transform=ax.transAxes, fontweight="bold", va="top")
    # the data runs corner to corner, so the guide label needs headroom to clear it
    lo, hi = ax.get_ylim()
    ax.set_ylim(lo, hi * 6)

fig.tight_layout()
fig.savefig(FIG_DIR / "convergence_refinement.pdf")

In [ ]:
# %% ===================== CELL C2 : convergence summary table =====================
# Roache/Celik verification metrics + the backward Euler prediction tau_num = tau + dt/2,
# i.e. a relative error dt/(2 tau) that the ratio below should reproduce.
dt, _, tau_t = cs_sweep("dt")
_, dx, tau_x = cs_sweep("dx")
ratio = (tau_t[-1] / TAU_INF_T - 1) / (dt[-1] / (TAU_INF_T * HOURS_TO_SEC))

conv_df = pd.DataFrame([
    {"quantity": "observed order of convergence $p$",
     "time step": f"{P_T:.3f}", "mesh": f"{P_X:.3f}"},
    {"quantity": r"Richardson extrapolate $\tau_\infty$ [h]",
     "time step": f"{TAU_INF_T:.4f}", "mesh": f"{TAU_INF_X:.4f}"},
    {"quantity": r"GCI on the finest discretisation [\%]",
     "time step": f"{GCI_T * 100:.3f}", "mesh": f"{GCI_X * 100:.1e}"},
    {"quantity": "finest discretisation",
     "time step": f"$\\Delta t / \\tau = {dt[-1] / (TAU_ANA * HOURS_TO_SEC):.1e}$",
     "mesh": f"$\\Delta x = {dx[-1]:.1e}$ m"},
    {"quantity": r"error on $\tau_\infty$ there [\%]",
     "time step": f"{(tau_t[-1] / TAU_INF_T - 1) * 100:.3f}",
     "mesh": f"{(tau_x[-1] / TAU_INF_X - 1) * 100:.1e}"},
]).set_index("quantity")
print(conv_df.to_string())
print(f"\nanalytical tau = {TAU_ANA:.4f} h -> tau_inf/tau_analytical - 1 = "
      f"{TAU_INF_T / TAU_ANA - 1:+.2e}")
print(f"measured [tau_fit(dt)/tau_inf - 1] / (dt/tau) = {ratio:.3f}  (backward Euler predicts 0.5)")

# LaTeX fragment for \input in the thesis (booktabs)
_ltx = conv_df.to_latex(
    escape=False, column_format="lrr",
    caption=(f"Grid convergence of the fitted decay time $\\tau_\\mathrm{{fit}}$ on the nominal "
             f"LIBRA-Pi input ($\\langle\\Pi\\rangle = {CS_META['Pi']:.3f}$), refinement ratio "
             f"$r = {R_REF:g}$, Roache safety factor $\\mathrm{{F_s}} = 1.25$."),
    label="tab:convergence_gci", position="htbp",
)
(FIG_DIR / "convergence_table.tex").write_text(_ltx)
print("wrote", FIG_DIR / "convergence_table.tex")

In [ ]:
# %% ===================== CELL A0 : load the analytical validity studies =====================
""" Validity of the analytical extraction time against the 1D ARD model, over a sampled space,
    correlated with the groups of the three assumptions (Pi, G_mix, G_P).
    Two designs are loaded and every figure below is produced for both:
      - analytical_validity  : h_l/K_s multiplier and tank height H  -> figures in thesis_fig/
      - design_space : K_s, P_top and E_l, one per group, at fixed geometry """
AV2_DIR = Path("data/design_space")

BASE_COLOR = "#4477AA"
SPP_COLOR = "0.6"
FLAG_EDGE_COLOR = "#CC3311"
BASE_EDGE_COLOR = "0.25"


def load_validity(run_dir: Path, csv_name: str, fig_dir: Path) -> dict:
    """Load one validity study and derive the discretisation-free error columns. No threshold is
    applied here: they are postprocessing choices, set in the CONTROLS cell and applied at plot
    time, so changing one only requires re-running the figure cells."""
    meta = json.load(open(run_dir / "metadata.json"))
    df = pd.read_csv(run_dir / csv_name)
    # every figure is produced twice: from the raw fitted tau, and with the backward Euler
    # bias dt/2 removed from it, which leaves the model error alone (suffix "_nodisc")
    df["tau_fit_nodisc_s"] = df.tau_fitted_s - df.dt_s / 2
    for tag, pred in (("pred", "tau_pred_s"), ("ave", "tau_pred_SPP_s"), ("bot", "tau_pred_bot_s")):
        df[f"e_{tag}_nodisc"] = (df.tau_fit_nodisc_s - df[pred]) / df[pred]
    fig_dir.mkdir(exist_ok=True, parents=True)
    return dict(df=df, meta=meta, fig=fig_dir, disc_error=meta["dt_fraction_of_tau"] / 2)


VALIDITY_STUDIES = {
    "design_space": load_validity(AV2_DIR, "design_space_data.csv", FIG_DIR),
}

for _name, _s in VALIDITY_STUDIES.items():
    print(f"{_name}: {len(_s['df'])} samples, {int(flagged_mask(_s['df']).sum())} non-exponential "
          f"at the current RMSE threshold, discretisation floor {_s['disc_error']:.1e} -> {_s['fig']}")

ERROR_SETS = {
    "": dict(tau_fit="tau_fitted_s", e_pred="e_pred", e_SPP="e_SPP", e_bot="e_bot",
             tau_lab=r"\tau_{\mathrm{fitted}}", disc=True),
    "_nodisc": dict(tau_fit="tau_fit_nodisc_s", e_pred="e_pred_nodisc", e_SPP="e_ave_nodisc",
                    e_bot="e_bot_nodisc", tau_lab=r"\tau_{\mathrm{fitted}} - \Delta t/2",
                    disc=False),
}

FILL_ALPHA = {True: 0.85, False: 0.15}


def scatter_samples(ax, x, y, df, x_group, color=BASE_COLOR, flag_nonexp=True):
    """Two independent encodings, so neither hides the other:
      fill opacity -- opaque when the two governing groups that are NOT on the x axis are both
                      below their own GROUP_THRESHOLDS entry, i.e. the trend against x is not
                      polluted by them;
      edge colour  -- red when the decay is not exponential (fit RMSE > RMSE_THRESHOLD), so that
                      tau_fitted is meaningless. The edge is always drawn opaque, including on
                      faded points, which is why the alpha goes in the face colour and not in
                      the `alpha` argument (that one would fade the edge too).
    Both masks are recomputed here from the CONTROLS cell, so re-running this cell after changing
    a threshold is enough."""
    from matplotlib.colors import to_rgba

    x, y = np.asarray(x, float), np.asarray(y, float)
    clean = clean_mask(df, x_group)
    # some figures do not use the exponential/non-exponential distinction; there the red edge
    # is noise, so it can be switched off
    flagged = flagged_mask(df) if flag_nonexp else np.zeros(len(df), bool)
    for cl in (False, True):
        for fl in (False, True):
            m = (clean == cl) & (flagged == fl)
            if not m.any():
                continue
            ax.scatter(x[m], y[m], marker="o", s=22, linewidths=0.7,
                       facecolors=to_rgba(color, FILL_ALPHA[cl]),
                       edgecolors=FLAG_EDGE_COLOR if fl else BASE_EDGE_COLOR,
                       zorder=2 + cl + 2 * fl)


def _sci(v):
    """1e-05 -> '10^{-5}' for mathtext labels."""
    mant, exp = f"{v:e}".split("e")
    mant, exp = float(mant), int(exp)
    return rf"10^{{{exp}}}" if abs(mant - 1) < 1e-12 else rf"{mant:g}\times 10^{{{exp}}}"


def sample_handles(x_group, color=BASE_COLOR, label=None, flag_nonexp=True):
    """Proxy artists for the two independent encodings of scatter_samples: fill = whether the two
    groups not on the x axis are below their thresholds, edge = whether the decay is exponential.
    Labels are rebuilt from the CONTROLS cell each time. The last handle keeps a faded fill on
    purpose: only its edge carries meaning. `label` overrides the first entry, for figures whose
    fill colour encodes a series rather than the clean/other split."""
    from matplotlib.colors import to_rgba
    from matplotlib.lines import Line2D
    mk = dict(marker="o", linestyle="none", markersize=4.5)
    handles = [
        Line2D([0], [0], **mk, markerfacecolor=to_rgba(color, FILL_ALPHA[True]),
               markeredgecolor=BASE_EDGE_COLOR,
               label=label if label is not None else others_label(x_group)),
        Line2D([0], [0], **mk, markerfacecolor=to_rgba(color, FILL_ALPHA[False]),
               markeredgecolor=BASE_EDGE_COLOR, label="otherwise"),
    ]
    if flag_nonexp:
        handles.append(
            Line2D([0], [0], **mk, markerfacecolor=to_rgba(color, FILL_ALPHA[False]),
                   markeredgecolor=FLAG_EDGE_COLOR,
                   label=rf"$\mathrm{{RMSE}}_\mathrm{{fit}} > {_sci(RMSE_THRESHOLD)}$"))
    return handles


def add_error_guides(ax, disc=None, signed=True, label_x=0.98):
    """Horizontal guides: y=0, +-5% (dotted) and, when `disc` is a value, the labelled
    discretisation floor dt/(2 tau) at +-disc."""
    if signed:
        ax.axhline(0, color="0.6", lw=0.6)
    for s in ((1, -1) if signed else (1,)):
        ax.axhline(s * 0.05, color="0.5", lw=0.7, ls=":")
        if disc:
            ax.axhline(s * disc, color="0.5", lw=0.7, ls="--")
    if disc:
        ax.text(label_x, disc, "discretization error", transform=ax.get_yaxis_transform(),
                fontsize=5.5, color="0.35", ha="right" if label_x > 0.5 else "left", va="center",
                bbox=dict(facecolor=plt.rcParams["axes.facecolor"], edgecolor="none", pad=0.5))

In [ ]:
# %% ===================== CELL A1 : parity plot =====================
# fitted decay time vs the analytical tau; y = x with a +-10% band, one dot per sample.
for study, S in VALIDITY_STUDIES.items():
    AV, AV_FLAGGED = S["df"], flagged_mask(S["df"])
    for tag, s in ERROR_SETS.items():
        fig, ax = plt.subplots(figsize=(3.5, 2.8))

        x = AV.tau_pred_s.to_numpy() * SEC_TO_HOURS
        y = AV[s["tau_fit"]].to_numpy() * SEC_TO_HOURS
        from matplotlib.colors import to_rgba
        for fl in (False, True):                       # edge stays opaque on faded fills
            m = AV_FLAGGED == fl
            if m.any():
                ax.scatter(x[m], y[m], s=16, linewidths=0.6, zorder=3 + fl,
                           facecolors=to_rgba(BASE_COLOR, 0.7),
                           edgecolors=FLAG_EDGE_COLOR if fl else BASE_EDGE_COLOR)

        lims = [min(x.min(), y.min()) * 0.7, max(x.max(), y.max()) * 1.4]
        ref = np.array(lims)
        ax.plot(ref, ref, color="0.3", lw=1.0, zorder=1, label=r"$y=x$")
        ax.plot(ref, 1.1 * ref, color="0.3", lw=0.7, ls="--", zorder=1, label=r"$\pm 10\%$")
        ax.plot(ref, 0.9 * ref, color="0.3", lw=0.7, ls="--", zorder=1)

        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.set_xlim(lims)
        ax.set_ylim(lims)
        ax.set_xlabel(r"$\tau\ \mathrm{[h]}$ (analytical)")
        ax.set_ylabel(rf"${s['tau_lab']}\ \mathrm{{[h]}}$ (1D ARD)")
        ax.legend(loc="upper left", fontsize=7)

        frac = float((np.abs(y - x) / x <= 0.10).mean())
        ax.text(0.97, 0.05, f"{frac:.0%} within " + r"$\pm10\%$", transform=ax.transAxes,
                fontsize=7, ha="right", color="0.2")

        fig.tight_layout()
        fig.savefig(S["fig"] / f"validity_parity{tag}.pdf")
        print(f"{study} parity{tag or ' (raw)'}: {frac:.1%} of samples within +-10% (n={len(AV)})")

In [ ]:
# %% ===================== CELL A2 : |error| vs Pi, SPP vs corrected tau =====================
# absolute error of the SPP extraction time (grey) and of the corrected one (blue), which
# includes the Pi/(1-exp(-Pi)) saturation factor.
for study, S in VALIDITY_STUDIES.items():
    AV, AV_FLAGGED = S["df"], flagged_mask(S["df"])
    for tag, s in ERROR_SETS.items():
        fig, ax = plt.subplots(figsize=(3.5, 2.8))

        for col, color in [(s["e_SPP"], SPP_COLOR), (s["e_pred"], BASE_COLOR)]:
            scatter_samples(ax, AV.Pi, AV[col].abs(), AV, "Pi", color=color)

        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.set_xlabel(r"$\langle\Pi\rangle$")
        ax.set_ylabel(rf"$|e| = |{s['tau_lab']}-\tau|\,/\,\tau$")
        add_error_guides(ax, disc=S["disc_error"] if s["disc"] else None, signed=False)
        ax.text(0.02, 0.05, r"$5\%$", transform=ax.get_yaxis_transform(), fontsize=5.5,
                color="0.35", ha="left", va="bottom")

        series = ax.legend(handles=sample_handles("Pi", SPP_COLOR, label=r"$\tau_{\mathrm{SPP}}$")[:1]
                           + sample_handles("Pi", label=r"$\tau = \tau_{\mathrm{SPP}}\,\Pi/(1-e^{-\Pi})$")[:1],
                           loc="upper left", fontsize=6, handlelength=1.2, labelspacing=0.3)
        ax.add_artist(series)
        ax.legend(handles=sample_handles("Pi"), loc="lower right", fontsize=5.5,
                  handlelength=1.2, labelspacing=0.3)

        fig.tight_layout()
        fig.savefig(S["fig"] / f"validity_abs_error_vs_Pi{tag}.pdf")
        hi = AV.Pi > 1
        for name, col in [("tau_SPP", s["e_SPP"]), ("tau corrected", s["e_pred"])]:
            print(f"{study:21s} {tag or '(raw)':9s} {name:14s}: median |e| = {AV[col].abs().median():.4f} "
                  f"(Pi>1: {AV[col][hi].abs().median():.3f}), "
                  f"{(AV[col].abs() <= 0.10).mean():.0%} within +-10%")

In [ ]:
# %% ===================== CELL A3+A4 : signed error vs G_mix and vs Pi =====================
# One figure, two panels sharing the y axis: the same signed error read against the two groups
# it can depend on. Merged from the two single-panel versions so the pair cannot drift apart
# and only one legend is needed.
for study, S in VALIDITY_STUDIES.items():
    AV, AV_FLAGGED = S["df"], flagged_mask(S["df"])
    for tag, s in ERROR_SETS.items():
        fig, (axm, axp) = plt.subplots(1, 2, figsize=(7.0, 3.0), sharey=True)

        scatter_samples(axm, AV.G_mix, AV[s["e_pred"]], AV, "G_mix")
        scatter_samples(axp, AV.Pi, AV[s["e_pred"]], AV, "Pi")

        axm.set_xlabel(r"$\Gamma_{\mathrm{mix}} = (H^2/E_l)\,/\,\tau_{\mathrm{fitted}}$")
        axp.set_xlabel(r"$\langle\Pi\rangle$")
        axm.set_ylabel(rf"$e = \dfrac{{{s['tau_lab']}-\tau}}{{\tau}}$")
        for ax, lab in ((axm, "(a)"), (axp, "(b)")):
            ax.set_xscale("log")
            ax.set_yscale("symlog", linthresh=0.01)
            ax.text(0.03, 0.94, lab, transform=ax.transAxes, fontweight="bold", va="top")
            add_error_guides(ax, disc=S["disc_error"] if s["disc"] else None)
        # the encoding is the same in both panels, only the pair of "other" groups differs;
        # one legend below the figure keeps it off the data, which is dense in every corner
        from matplotlib.lines import Line2D
        from matplotlib.colors import to_rgba
        mk = dict(marker="o", linestyle="none", markersize=4.5)
        shared = [
            Line2D([0], [0], **mk, markerfacecolor=to_rgba(BASE_COLOR, FILL_ALPHA[True]),
                   markeredgecolor=BASE_EDGE_COLOR,
                   label="the two groups not on the $x$ axis are both $< 0.1$"),
            Line2D([0], [0], **mk, markerfacecolor=to_rgba(BASE_COLOR, FILL_ALPHA[False]),
                   markeredgecolor=BASE_EDGE_COLOR, label="otherwise"),
            Line2D([0], [0], **mk, markerfacecolor=to_rgba(BASE_COLOR, FILL_ALPHA[False]),
                   markeredgecolor=FLAG_EDGE_COLOR,
                   label=rf"$\mathrm{{RMSE}}_\mathrm{{fit}} > {_sci(RMSE_THRESHOLD)}$"),
        ]
        fig.legend(handles=shared, loc="lower center", ncol=3, fontsize=6.5,
                   handlelength=1.2, columnspacing=1.6, bbox_to_anchor=(0.5, -0.05))

        fig.tight_layout()
        fig.savefig(S["fig"] / f"validity_error_vs_groups{tag}.pdf")


In [ ]:
# %% (folded into the cell above: the two panels are now one figure)


In [ ]:
# %% ===================== CELL A5 : error vs G_P =====================
# left = bottom-evaluated tau_0, right = the corrected height-averaged tau
for study, S in VALIDITY_STUDIES.items():
    AV, AV_FLAGGED = S["df"], flagged_mask(S["df"])
    for tag, s in ERROR_SETS.items():
        fig, (axb, axa) = plt.subplots(1, 2, figsize=(7.0, 3.0), sharex=True, sharey=True)

        scatter_samples(axb, AV.G_P, AV[s["e_bot"]], AV, "G_P", flag_nonexp=False)
        scatter_samples(axa, AV.G_P, AV[s["e_pred"]], AV, "G_P", flag_nonexp=False)

        axb.set_ylabel(rf"$e_0 = \dfrac{{{s['tau_lab']}-\tau_0}}{{\tau_0}}$")
        axa.set_ylabel(rf"$e = \dfrac{{{s['tau_lab']}-\tau}}{{\tau}}$")
        for ax, title, lab in [(axb, r"bottom-evaluated $\tau_0$", "(a)"),
                               (axa, r"height-averaged $\tau$", "(b)")]:
            ax.set_xscale("log")
            ax.set_yscale("symlog", linthresh=0.01)
            ax.set_xlabel(r"$\Gamma_P$")
            ax.set_title(title, fontsize=9)
            ax.text(0.03, 0.94, lab, transform=ax.transAxes, fontweight="bold", va="top")
            add_error_guides(ax, disc=S["disc_error"] if s["disc"] else None)
        # below the panels: the cloud fills every corner of (a), so an inset legend would
        # always sit on data
        fig.legend(handles=sample_handles("G_P", flag_nonexp=False), loc="lower center",
                   ncol=2, fontsize=6.5, handlelength=1.2, columnspacing=1.6,
                   bbox_to_anchor=(0.5, -0.04))

        fig.tight_layout()
        fig.savefig(S["fig"] / f"validity_error_vs_GP{tag}.pdf")

In [ ]:
# %% ===================== CELL A6 : text summary =====================
# for each design: how it sampled the space, then (i) the effect of each group alone, on the
# population where the OTHER two are small, and (ii) the joint effect of Pi and G_P, which is
# where the analytical tau actually breaks down.
PI_BINS = [(0, 0.1), (0.1, 1), (1, 10), (10, np.inf)]
GP_BINS = [(0, 0.1), (0.1, 1), (1, np.inf)]

print("thresholds applied (CONTROLS cell): RMSE > "
      f"{RMSE_THRESHOLD:g} flags a non-exponential decay; per-group "
      + ", ".join(f"{g} < {t:g}" for g, t in GROUP_THRESHOLDS.items()))

for study, S in VALIDITY_STUDIES.items():
    AV, AV_FLAGGED = S["df"], flagged_mask(S["df"])
    ok = ~AV_FLAGGED
    groups = {"Pi": AV.Pi.to_numpy(), "G_mix": AV.G_mix.to_numpy(), "G_P": AV.G_P.to_numpy()}

    print(f"\n{'=' * 78}\n{study}: {len(AV)} samples, {int(AV_FLAGGED.sum())} non-exponential, "
          f"discretisation floor {S['disc_error']:.1e}\n{'=' * 78}")
    print("sampling: " + S["meta"].get("description", "see metadata.json"))
    print("--- space covered, and how correlated the groups came out ---")
    for name, g in groups.items():
        print(f"{name:6s}: {g.min():.2e} .. {g.max():.2e}")
    lg = {k: np.log10(v) for k, v in groups.items()}
    for a, b in (("Pi", "G_P"), ("Pi", "G_mix"), ("G_P", "G_mix")):
        print(f"   corr(log {a}, log {b}) = {np.corrcoef(lg[a], lg[b])[0, 1]:+.3f}")
    if "eps_gH" in AV:
        limit = S["meta"].get("no_coalescence_eps_g_limit", 0.01)
        print(f"eps_g(H): {AV.eps_gH.min():.2e} .. {AV.eps_gH.max():.2e}; "
              f"{(AV.eps_gH > limit).sum()}/{len(AV)} above the {limit:.0%} no-coalescence limit")

    for tag, s in ERROR_SETS.items():
        e = AV[s["e_pred"]].to_numpy()
        print(f"\n--- {'raw tau_fitted' if not tag else 'tau_fitted - dt/2'} ---")
        print(f"overall: median |e| = {np.median(np.abs(e)):.2e}, "
              f"{(np.abs(e) <= 0.05).mean():.0%} within +-5%, {(np.abs(e) <= 0.10).mean():.0%} within +-10%")
        print(f"SPP tau: median |e| = {AV[s['e_SPP']].abs().median():.2e}, "
              f"{(AV[s['e_SPP']].abs() <= 0.10).mean():.0%} within +-10%")
        print(f"signed error range: {e.min():+.3f} .. {e.max():+.3f}")

        print("each group alone (the other two below their own threshold, non-flagged):")
        for name, g in groups.items():
            clean = clean_mask(AV, name) & ok
            if not clean.any():
                print(f"   {name:6s}: no sample satisfies {others_label(name)}")
                continue
            print(f"   {name:6s}: n={int(clean.sum()):3d}, spans {g[clean].min():.1e}..{g[clean].max():.1e}, "
                  f"median |e| = {np.median(np.abs(e[clean])):.2e}, max |e| = {np.max(np.abs(e[clean])):.2e}")

        print("median signed error on a Pi x G_P grid (non-flagged, n in parentheses):")
        print(f"{'Pi \\ G_P':>12s}" + "".join(f"{f'{a:g}-{b:g}':>16s}" for a, b in GP_BINS))
        for pa, pb in PI_BINS:
            row = f"{f'{pa:g}-{pb:g}':>12s}"
            for ga, gb in GP_BINS:
                m = (ok & (groups["Pi"] >= pa) & (groups["Pi"] < pb)
                     & (groups["G_P"] >= ga) & (groups["G_P"] < gb))
                row += f"{(f'{np.median(e[m]):+.3f} ({m.sum()})' if m.any() else '-'):>16s}"
            print(row)

In [ ]:
# %% (removed) the factorial study is not used in the thesis and its dataset is not
# published; the design-space questions it explored are answered by the Venn, the
# violin and the parity figures built from analytical_validity2.


In [ ]:
# %% (removed) the factorial study is not used in the thesis and its dataset is not
# published; the design-space questions it explored are answered by the Venn, the
# violin and the parity figures built from analytical_validity2.


In [ ]:
# %% (removed) the factorial study is not used in the thesis and its dataset is not
# published; the design-space questions it explored are answered by the Venn, the
# violin and the parity figures built from analytical_validity2.


In [ ]:
# %% (removed) the factorial study is not used in the thesis and its dataset is not
# published; the design-space questions it explored are answered by the Venn, the
# violin and the parity figures built from analytical_validity2.


In [ ]:
# %% ===================== CELL R1 : SPP and PPL limiting regimes =====================
# tau_fitted / tau_SPP against <Pi>, with the analytical factor Pi/(1-exp(-Pi)) and its two
# asymptotes. Where the points sit on the lower asymptote the system is mass transfer limited
# (SPP), where they sit on the upper one it is partial pressure limited (PPL).
#
# Restricted to the samples where the two assumptions that are NOT about saturation hold
# (G_mix < 0.1 and G_P < 0.1), so the spread around the curve is the effect of <Pi> alone.
# None of those samples is flagged as non-exponential, so the fit-quality encoding used
# elsewhere carries no information here and a plain scatter is used instead.
from matplotlib.colors import to_rgba

S = VALIDITY_STUDIES["design_space"]
AV = S["df"]
keep = ((AV.G_mix < GROUP_THRESHOLDS["G_mix"]) & (AV.G_P < GROUP_THRESHOLDS["G_P"])).to_numpy()
assert not flagged_mask(AV)[keep].any(), "a flagged sample survived the filter"

tau_sim = AV.tau_fit_nodisc_s.to_numpy()[keep]
tau_spp = AV.tau_pred_SPP_s.to_numpy()[keep]
Pi = AV.Pi.to_numpy()[keep]
ratio = tau_sim / tau_spp

fig, ax = plt.subplots(figsize=(3.5, 2.8))
ax.scatter(Pi, ratio, marker="o", s=22, linewidths=0.7,
           facecolors=to_rgba(BASE_COLOR, FILL_ALPHA[True]), edgecolors=BASE_EDGE_COLOR,
           zorder=3, label=rf"samples, $\Gamma_\mathrm{{mix}}<0.1$ and $\Gamma_P<0.1$")

pi_line = np.logspace(np.log10(Pi.min()), np.log10(Pi.max()), 300)
ax.plot(pi_line, -pi_line / np.expm1(-pi_line), color="0.15", lw=1.1, zorder=5,
        label=r"$\Pi/(1-e^{-\Pi})$")
ax.axhline(1, color="#4477AA", lw=0.8, ls="--", zorder=4,
           label=r"SPP asymptote, $\tau \to \tau_{\mathrm{SPP}}$")
ax.plot(pi_line, pi_line, color="#EE7733", lw=0.8, ls="--", zorder=4,
        label=r"PPL asymptote, $\tau \to \tau_{\mathrm{SPP}}\,\langle\Pi\rangle$")

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_ylim(0.7, ratio.max() * 2)          # the PPL asymptote runs off below; clip to the data
ax.set_xlabel(r"$\langle\Pi\rangle$")
ax.set_ylabel(r"$\tau_{\mathrm{fitted}} / \tau_{\mathrm{SPP}}$")
ax.legend(loc="upper left", fontsize=6)
fig.tight_layout()
fig.savefig(S["fig"] / "regimes_tau_vs_Pi.pdf")

print(f"regimes figure: {int(keep.sum())} of {len(AV)} samples kept "
      f"(G_mix < {GROUP_THRESHOLDS['G_mix']}, G_P < {GROUP_THRESHOLDS['G_P']})")
# where does each asymptote hold to better than 5%?
for name, r in (("SPP", ratio), (r"PPL (tau_SPP*Pi)", tau_sim / (tau_spp * Pi))):
    ok5 = np.abs(r - 1) < 0.05
    if ok5.any():
        print(f"   {name:18s} holds within 5% for <Pi> in {Pi[ok5].min():.3g} .. "
              f"{Pi[ok5].max():.3g}  (n={int(ok5.sum())})")


In [ ]:
# %% ===================== CELL D1 : design space, effect of the top pressure =====================
# At fixed molar gas flow, lowering P_top expands the bubbles: eps_g and the interfacial area
# grow, so the extraction gets faster. Restricted to the SPP samples, where tau does not depend
# on K_s, so the trend is the hydrodynamic effect alone. Shaded: where eps_g(H) exceeds the 1%
# void fraction behind the "no bubble coalescence" assumption, i.e. where the model is
# extrapolating beyond its own premises.
S = VALIDITY_STUDIES["design_space"]
AV, AV_FLAGGED = S["df"], flagged_mask(S["df"])
EPS_LIMIT = S["meta"].get("no_coalescence_eps_g_limit", 0.01)

spp = (AV.Pi < GROUP_THRESHOLDS["Pi"]).to_numpy() & ~AV_FLAGGED
p_atm = (AV.P_top_Pa / 101325).to_numpy()
tau_h = AV.tau_fit_nodisc_s.to_numpy() * SEC_TO_HOURS

fig, ax = plt.subplots(figsize=(3.5, 2.8))
bad = (AV.eps_gH > EPS_LIMIT).to_numpy()
if bad.any():
    ax.axvspan(p_atm.min() * 0.7, p_atm[bad].max(), color=FLAG_EDGE_COLOR, alpha=0.08, zorder=0)
    # anchored at the top of the band: tau rises with P_top, so the data sits low on the left
    ax.text(p_atm[bad].max(), 0.97, r" $\varepsilon_g(H) > 1\%$, coalescence likely",
            transform=ax.get_xaxis_transform(), fontsize=5.5, color=FLAG_EDGE_COLOR,
            va="top", ha="right", rotation=90)

ax.scatter(p_atm[~spp], tau_h[~spp], s=14, facecolors="0.75", edgecolors="none",
           alpha=0.5, zorder=2, label=r"$\langle\Pi\rangle \geq 0.1$")
ax.scatter(p_atm[spp], tau_h[spp], s=18, facecolors=BASE_COLOR, edgecolors=BASE_EDGE_COLOR,
           linewidths=0.6, alpha=0.85, zorder=3, label=r"$\langle\Pi\rangle < 0.1$ (SPP)")

slope, intercept = np.polyfit(np.log10(p_atm[spp]), np.log10(tau_h[spp]), 1)
pl = np.logspace(np.log10(p_atm.min()), np.log10(p_atm.max()), 50)
ax.plot(pl, 10**intercept * pl**slope, color="0.15", lw=1.0, ls=(0, (3, 2)), zorder=5,
        label=rf"$\tau \propto P_{{\mathrm{{top}}}}^{{{slope:.2f}}}$")

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlim(p_atm.min() * 0.7, p_atm.max() * 1.4)
ax.set_xlabel(r"$P_{\mathrm{top}}\ \mathrm{[atm]}$")
ax.set_ylabel(r"$\tau_{\mathrm{fitted}}\ \mathrm{[h]}$")
ax.legend(loc="lower right", fontsize=6)
fig.tight_layout()
fig.savefig(S["fig"] / "design_tau_vs_Ptop.pdf")

print(f"SPP subset (n={int(spp.sum())}): tau ~ P_top^{slope:.3f}")
lo, hi = p_atm[spp].min(), p_atm[spp].max()
print(f"   over P_top {lo:.3g} .. {hi:.3g} atm, tau ranges {tau_h[spp].min():.3g} .. {tau_h[spp].max():.3g} h")
ok = spp & ~bad
if ok.any():
    s_ok = np.polyfit(np.log10(p_atm[ok]), np.log10(tau_h[ok]), 1)[0]
    print(f"   restricted to eps_g(H) < {EPS_LIMIT:.0%} (n={int(ok.sum())}): exponent {s_ok:.3f}")
print(f"   d_b(0) spans {AV.d_b0_m.min() * 1e3:.2f} .. {AV.d_b0_m.max() * 1e3:.2f} mm "
      f"(bubble diameter correlation is extrapolated at both ends of the P_top range)")

In [ ]:
# %% ===================== CELL N0 : load the single-sample re-runs =====================
""" Two design_space samples re-run with every field exported (generate_data.non_exponential_case
    and .stratified_case), each in a corner where the analytical solution fails differently:
      non_exponential : G_mix >> 1 at Pi >> 1  -> saturated bubbles, decay no longer exponential
      stratified      : G_mix >> 1 and G_P >> 1 -> the liquid stratifies, extraction gets slower """
SAMPLE_STUDIES = {}
for _d in ("data/non_exponential_sample", "data/stratified_sample"):
    _d = Path(_d)
    meta = json.load(open(_d / "metadata.json"))
    (_d / "fig").mkdir(exist_ok=True, parents=True)
    SAMPLE_STUDIES[_d.name] = dict(
        meta=meta, rec=meta["record"], fig=_d / "fig", tau=meta["record"]["tau_pred_s"],
        ard=pd.read_csv(_d / "ard_inputs.csv").set_index("key"),
        exports={n: load_export(_d / f"{n}.csv")
                 for n in ("c_T2", "P_T2", "aJ_T2", "n_T2_salt", "a", "h_l", "u_g", "eps_l")},
    )

for _name, _s in SAMPLE_STUDIES.items():
    r = _s["rec"]
    print(f"{_name} (sample {_s['meta']['sample_id']}): Pi={r['Pi']:.3g}, G_P={r['G_P']:.3g}, "
          f"G_mix={r['G_mix_pred']:.3g}")
    print(f"   tau_pred={r['tau_pred_s'] / HOURS_TO_SEC:.2f} h, "
          f"tau_fitted={r['tau_fitted_s'] / HOURS_TO_SEC:.2f} h "
          f"({r['tau_fitted_s'] / r['tau_pred_s'] - 1:+.1%}), fit RMSE={r['fit_rmse_norm']:.3e}")


In [ ]:
# %% ===================== CELL N1 : one figure per single-sample re-run ==============
# (a) the profiles: with G_mix >> 1 the liquid cannot flatten its own profile over one
#     extraction time, so c_T2 keeps the shape the source term gives it.
# (b) the inventory and the residual of its exponential fit -- the residual is the only place
#     where the departure from a single exponential is visible.
from scipy.integrate import cumulative_trapezoid

from sparging.plot_style import COLORS
from sparging.postprocess import fit_exp

# the two analytical partial pressure profiles of the thesis: the compact one, which assumes
# uniform hydrodynamics, and the general one of the appendix, which does not
ANA_UNIF_KW = dict(color="0.15", lw=0.8, ls=(0, (2, 1.6)), zorder=5)
ANA_GEN_KW = dict(color=COLORS["red"], lw=0.8, ls=(0, (1, 1.4)), zorder=6)


R_GAS = 8.314462618      # J/mol/K


def _analytical_P_T2(E, K_s, T_K, c_mean):
    """The two analytical gas profiles, from the exported steady hydrodynamics.

        Pi*(z) = R T K_s int_0^z a h_l / u_g ds
        uniform : P = c/K_s (1 - exp(-Pi*))            -- a, h_l, u_g taken as constant
        general : P = R T c / u_g int_0^z a h_l exp(-(Pi*(z) - Pi*(s))) ds

    The difference between the two is what the uniform hydrodynamics assumption costs.
    """
    z, a, h_l, u_g = E["a"]["x"], E["a"]["data"], E["h_l"]["data"], E["u_g"]["data"]
    pi_star = R_GAS * T_K * K_s * cumulative_trapezoid(a * h_l / u_g, z, initial=0.0)
    uniform = c_mean / K_s * (1 - np.exp(-pi_star))
    inner = cumulative_trapezoid(a * h_l * np.exp(pi_star), z, initial=0.0)
    return uniform, R_GAS * T_K * c_mean / u_g * np.exp(-pi_star) * inner


def _liquid_mean(E, i):
    """<c_T2>^l at time index i: the intrinsic average, weighted by the liquid fraction."""
    z, eps_l = E["c_T2"]["x"], E["eps_l"]["data"]
    return np.trapezoid(E["c_T2"]["data"][i] * eps_l, z) / np.trapezoid(eps_l, z)


def sample_profile_figure(S, times, analytical=False):
    """The nonexp layout: three stacked profiles on the left, inventory and fit residual on the
    right. With `analytical`, the analytical P_T2 profile (eq. PT_profile) is overlaid on the
    pressure panel and the analytical decay is added to the inventory panel, so the departure
    from the analytical solution can be read on the fields and on the integral at once."""
    E, REC, TAU = S["exports"], S["rec"], S["tau"]

    fig = plt.figure(figsize=(7.0, 5.0))
    gs = fig.add_gridspec(3, 2, height_ratios=[1, 1, 1], width_ratios=[1, 1],
                          hspace=0.12, wspace=0.30)
    ax_c = fig.add_subplot(gs[0, 0])
    ax_p = fig.add_subplot(gs[1, 0], sharex=ax_c)
    ax_j = fig.add_subplot(gs[2, 0], sharex=ax_c)
    ax_n = fig.add_subplot(gs[0:2, 1])
    ax_r = fig.add_subplot(gs[2, 1], sharex=ax_n)

    # --- (a) profiles ---
    idx = select_nearest(E["c_T2"]["times"], times * TAU)
    colors = plt.cm.viridis(np.linspace(0.05, 0.8, len(idx)))
    for ax, name, ylabel in zip(
        (ax_c, ax_p, ax_j),
        ("c_T2", "P_T2", "aJ_T2"),
        (r"$c_{T_2}\ \mathrm{[mol\,m^{-3}]}$", r"$P_{T_2}\ \mathrm{[Pa]}$",
         r"$a\,J_{T_2}\ \mathrm{[mol\,m^{-3}\,s^{-1}]}$"),
    ):
        e = E[name]
        for col, i in zip(colors, idx):
            ax.plot(e["x"] * M_TO_CM, e["data"][i], color=col, lw=1.3,
                    label=fr"$t = {e['times'][i] / TAU:.2f}\,\tau$")
        ax.set_ylabel(ylabel, fontsize=8)
        ax.ticklabel_format(axis="y", style="sci", scilimits=(-2, 2))

    if analytical:
        K_s = float(S["ard"].loc["K_s", "value"])
        T_K = float(S["ard"].loc["T", "value"]) + 273.15
        for i in idx:
            unif, gen = _analytical_P_T2(E, K_s, T_K, _liquid_mean(E, i))
            ax_p.plot(E["P_T2"]["x"] * M_TO_CM, unif, **ANA_UNIF_KW)
            ax_p.plot(E["P_T2"]["x"] * M_TO_CM, gen, **ANA_GEN_KW)
        ax_p.plot([], [], **ANA_UNIF_KW, label="analytical, uniform")
        ax_p.plot([], [], **ANA_GEN_KW, label="analytical, general")
        ax_p.legend(loc="upper left", fontsize=6)

    for ax in (ax_c, ax_p):
        ax.tick_params(labelbottom=False)
    ax_j.legend(loc="upper right", fontsize=6)
    ax_j.set_xlabel(r"$z\ \mathrm{[cm]}$")
    ax_j.set_xlim(0, E["c_T2"]["x"].max() * M_TO_CM)

    # --- (b) inventory and residual ---
    t, n = E["n_T2_salt"]["times"], E["n_T2_salt"]["data"]
    (tau_fit, n0_fit), _ = fit_exp(n * ureg.molT2, t * ureg.s, t[0] * ureg.s, t[-1] * ureg.s,
                                   "decay", tau_guess=TAU * ureg.s)
    tau_fit_s = tau_fit.to("s").magnitude
    n_fit = n0_fit.to("molT2").magnitude * np.exp(-t / tau_fit_s)

    ax_n.plot(t / TAU, n, color=BASE_COLOR, **SIM_KW, label="1D ARD model")
    ax_n.plot(t / TAU, n_fit, **ANA_KW,
              label=rf"fit, $\tau_\mathrm{{num}} = {tau_fit_s / HOURS_TO_SEC:.2f}$ h")
    if analytical:
        ax_n.plot(t / TAU, n[0] * np.exp(-t / TAU), color=COLORS["red"], lw=0.9, ls=":",
                  zorder=4, label=rf"analytical, $\tau_\mathrm{{ana}} = "
                                  rf"{TAU / HOURS_TO_SEC:.2f}$ h")
    ax_n.set_yscale("log")
    ax_n.set_ylabel(r"$n_{T_2}\ \mathrm{[mol]}$")
    ax_n.legend(loc="lower left", fontsize=7)
    ax_n.tick_params(labelbottom=False)

    ax_r.plot(t / TAU, (n - n_fit) / n_fit * 100, color=BASE_COLOR, lw=1.1)
    ax_r.axhline(0, color="0.6", lw=0.6)
    ax_r.set_xlabel(r"$t/\tau_\mathrm{ana}$")
    ax_r.set_ylabel("residual [%]", fontsize=8)

    for ax, lab in ((ax_c, "(a)"), (ax_n, "(b)")):
        ax.text(0.02, 0.94, lab, transform=ax.transAxes, fontweight="bold", va="top",
                fontsize=9)

    res = (n - n_fit) / n_fit
    print(f"fit tau = {tau_fit_s / HOURS_TO_SEC:.4f} h vs analytical {TAU / HOURS_TO_SEC:.4f} h "
          f"({tau_fit_s / TAU - 1:+.3%}); residual max |.| = {np.abs(res).max():.2%}; "
          f"normalized RMSE = {REC['fit_rmse_norm']:.3e}")
    return fig


SAMPLE_FIGURES = {
    # study directory -> (output name, profile times in units of tau, overlay the analytical)
    "non_exponential_sample": ("nonexp.pdf", np.array([0.05, 0.2, 0.5, 1.0, 2.0]), False),
    "stratified_sample": ("stratified.pdf", np.array([0.05, 0.5, 2.0]), True),
}
for _study, (_name, _times, _ana) in SAMPLE_FIGURES.items():
    print(f"--- {_study} -> {_name}")
    sample_profile_figure(SAMPLE_STUDIES[_study], _times, _ana).savefig(
        FIG_DIR / _name, bbox_inches="tight")


In [ ]:
# %% (folded into the cell above: profiles and inventory are now one figure)


In [ ]:
# %% ===================== DESIGN MAP: VENN, VIOLINS, PARITY =====================
# The design-space figures of the sparging design chapter.
# 
# All read the 300-sample design study and split it on the three governing groups. Each figure
# answers a different question, and they are built differently on purpose:
# 
#     venn     "given the region I operate in, which model may I use, and how far off is the
#               analytical solution?"      -> signed error of tau_ana per region
#     violin   "which region gives the fastest extraction?"
#               -> tau_num per region, against a reference (several normalisations, see below)
#     parity   "may I trust the analytical solution here?"
#               -> tau_num vs tau_ana, the two valid groups highlighted
# 
# The Venn circles are the *violated* assumptions, so its centre is the region where none of them
# holds, and each region carries the case number of the decision table (tab:design_map).
# 
# Run from the repository root: python3 make_design_map_figures.py

import copy
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from matplotlib.cm import ScalarMappable
from matplotlib.lines import Line2D
from matplotlib.path import Path as MplPath
from matplotlib.patches import FancyBboxPatch, PathPatch
from matplotlib_venn import venn3
from matplotlib_venn.layout.venn3 import DefaultLayoutAlgorithm

from sparging.plot_style import apply as apply_style, categorical_axis, COLORS


DATA_DIR = Path("data/design_space")
FIG_DIR = Path("figures")
THESIS_FIG = Path("figures")

RMSE_THRESHOLD = 1e-4      # above this the inventory decay is not a single exponential
NONEXP_HATCH_FRAC = 0.5    # hatch a region when this fraction of its samples are non-exponential
THRESH = 0.1               # common threshold on the three groups
PARITY_BAND = 0.05         # +-5 % band on the parity plot

# hatching: pale, so it reads as an annotation over the diverging fill rather than competing
VALID_HATCH, VALID_COLOR = "///", "#66BB6A"       # analytical solution trustworthy
NONEXP_HATCH, NONEXP_COLOR = "\\\\\\", "#EF8A80"    # decay is not a single exponential

# --- notation: matches \Gmix and \Gp in thesis.tex --------------------------- #
SYM_MIX = r"\Gamma_\mathrm{mix}"
SYM_P = r"\Gamma_P"
SYM_PI = r"\langle\Pi\rangle"

# regions keyed as (Pi, G_mix, G_P) VIOLATED flags -> case number in tab:design_map,
# which is indexed on the SATISFIED conditions (G_mix<0.1, G_P<0.1, Pi<0.1)
CASE_OF = {
    (0, 0, 0): 1, (1, 0, 0): 2, (0, 0, 1): 3, (1, 0, 1): 4,
    (0, 1, 0): 5, (1, 1, 0): 6, (0, 1, 1): 7, (1, 1, 1): 8,
}
REGION_NAME = {
    (0, 0, 0): "none", (1, 0, 0): rf"${SYM_PI}$", (0, 1, 0): rf"${SYM_MIX}$",
    (0, 0, 1): rf"${SYM_P}$", (1, 1, 0): rf"${SYM_PI}+{SYM_MIX}$",
    (1, 0, 1): rf"${SYM_PI}+{SYM_P}$", (0, 1, 1): rf"${SYM_MIX}+{SYM_P}$",
    (1, 1, 1): "all three",
}

apply_style()


def load():
    df = pd.read_csv(DATA_DIR / "design_space_data.csv", comment="#")
    # the fitted tau carries the backward Euler +dt/2 bias; removing it leaves the model error
    df["tau_fit_nodisc_s"] = df.tau_fitted_s - df.dt_s / 2
    df["e_pred_nodisc"] = (df.tau_fit_nodisc_s - df.tau_pred_s) / df.tau_pred_s
    df["nonexp"] = df.fit_rmse_norm >= RMSE_THRESHOLD
    df["vPi"] = (df.Pi > THRESH).astype(int)
    df["vmix"] = (df.G_mix > THRESH).astype(int)
    df["vP"] = (df.G_P > THRESH).astype(int)
    df["case"] = [CASE_OF[k] for k in zip(df.vPi, df.vmix, df.vP)]
    df["n_violated"] = df.vPi + df.vmix + df.vP
    return df


def region_mask(df, key):
    return (df.vPi == key[0]) & (df.vmix == key[1]) & (df.vP == key[2])


def region_stats(df):
    out = {}
    for key in REGION_NAME:
        s = df[region_mask(df, key)]
        out[key] = dict(name=REGION_NAME[key], case=CASE_OF[key], n=len(s),
                        median_e=s.e_pred_nodisc.median() if len(s) else np.nan,
                        range=_range_label(s.e_pred_nodisc) if len(s) else "",
                        nonexp=s.nonexp.mean() if len(s) else 0.0)
    return out


def text_on(rgba):
    """Black or white, whichever stays readable on `rgba` (ITU-R BT.601 luminance)."""
    r, g, b = rgba[:3]
    return "white" if (0.299 * r + 0.587 * g + 0.114 * b) < 0.5 else "0.1"


def pct(value):
    """Signed percentage, without the '-0.0%' that rounding to one decimal produces."""
    v = value * 100
    return "0.0%" if abs(v) < 0.05 else f"{v:+.1f}%"


# --------------------------------------------------------------------------- #
# Venn
# --------------------------------------------------------------------------- #

def _outside_path(x0, y0, x1, y1, centers, radii, n=240):
    """The frame minus the three circles, as one path.

    Built by winding the rectangle counter-clockwise and each circle clockwise, so the nonzero
    fill rule subtracts them. Circles are polygonised, which is invisible at this size and
    avoids reversing Bezier segments.
    """
    rect = np.array([[x0, y0], [x1, y0], [x1, y1], [x0, y1], [x0, y0]])
    verts, codes = [rect], [[MplPath.MOVETO] + [MplPath.LINETO] * 3 + [MplPath.CLOSEPOLY]]
    theta = np.linspace(0, 2 * np.pi, n)[::-1]          # reversed: clockwise
    for centre, r in zip(centers, radii):
        cx, cy = centre.x, centre.y
        pts = np.column_stack([cx + r * np.cos(theta), cy + r * np.sin(theta)])
        pts[-1] = pts[0]
        verts.append(pts)
        codes.append([MplPath.MOVETO] + [MplPath.LINETO] * (n - 2) + [MplPath.CLOSEPOLY])
    return MplPath(np.concatenate(verts), np.concatenate(codes))


def _range_label(s):
    """Observed spread of the signed error, as the designer would quote it."""
    lo, hi = s.min() * 100, s.max() * 100

    def fmt(v):
        if abs(v) < 0.05:      # avoids the "-0.0" that plain rounding produces
            return "0"
        return f"{v:.0f}" if abs(v) >= 1 else f"{v:.1f}"

    return f"{fmt(lo)} to {fmt(hi)}%"


def make_venn(stats, df):
    """Which model may I use, and how far off is the analytical solution.

    Fill colour is the median signed error, the label under each case number is the observed
    range of that error, which is what a designer needs: pick the analytical formula here and
    the true value sits somewhere in this interval. Green hatching marks where the analytical
    solution is good to +-5 % (at most one assumption violated), red where the decay is no
    longer a single exponential.
    """
    lim = max(abs(v["median_e"]) for v in stats.values())
    norm = TwoSlopeNorm(vmin=-lim, vcenter=0.0, vmax=lim)
    cmap = plt.get_cmap("RdBu_r")

    fig, ax = plt.subplots(figsize=(7.0, 4.4))
    v = venn3(
        subsets=(1, 1, 1, 1, 1, 1, 1),
        set_labels=(rf"${SYM_PI}>0.1$", rf"${SYM_MIX}>0.1$", rf"${SYM_P}>0.1$"),
        ax=ax,
        layout_algorithm=DefaultLayoutAlgorithm(fixed_subset_sizes=(1,) * 7),
    )

    outside = stats[(0, 0, 0)]
    for key, st in stats.items():
        if key == (0, 0, 0):
            continue
        pid = "".join(str(b) for b in key)
        patch, label = v.get_patch_by_id(pid), v.get_label_by_id(pid)
        if patch is None:
            continue
        patch.set_color(cmap(norm(st["median_e"])))
        patch.set_alpha(1.0)
        patch.set_edgecolor("0.3")
        patch.set_linewidth(0.7)
        overlay = None
        if sum(key) <= 1:
            overlay = (VALID_HATCH, VALID_COLOR)
        elif st["nonexp"] >= NONEXP_HATCH_FRAC:
            overlay = (NONEXP_HATCH, NONEXP_COLOR)
        if overlay is not None:
            hp = copy.copy(patch)
            hp.set_facecolor("none")
            hp.set_edgecolor(overlay[1])
            hp.set_hatch(overlay[0])
            hp.set_linewidth(0)
            ax.add_patch(hp)
        if label is not None:
            # the case number is what a designer looks up, so it carries the weight;
            # the error range sits under it in a lighter, smaller style
            label.set_text(f"$\\bf{{{st['case']}}}$\n{st['range']}")
            label.set_fontsize(9)
            label.set_color(text_on(cmap(norm(st["median_e"]))))
    for t in v.set_labels:
        if t is not None:
            t.set_fontsize(9)

    fig.canvas.draw()
    inv = ax.transData.inverted()
    xs, ys = [], []
    for t in list(v.set_labels) + list(v.subset_labels):
        if t is None:
            continue
        bb = t.get_window_extent(fig.canvas.get_renderer()).transformed(inv)
        xs += [bb.x0, bb.x1]; ys += [bb.y0, bb.y1]
    for patch in v.patches:
        if patch is None:
            continue
        ext = patch.get_path().get_extents(patch.get_transform()).transformed(inv)
        xs += [ext.x0, ext.x1]; ys += [ext.y0, ext.y1]
    pad = 0.10
    x0, x1, y0, y1 = min(xs) - pad, max(xs) + pad, min(ys) - pad, max(ys) + pad
    y0 -= 0.12                       # room for the case 1 label in the free corner

    ax.add_patch(FancyBboxPatch(
        (x0, y0), x1 - x0, y1 - y0, boxstyle="round,pad=0.005,rounding_size=0.03",
        linewidth=0.9, edgecolor="0.35", facecolor=cmap(norm(outside["median_e"])), zorder=-5))
    # case 1 is everything inside the frame and outside every circle: hatch exactly that, so
    # there is no doubt that the analytical solution is valid there too
    ax.add_patch(PathPatch(
        _outside_path(x0, y0, x1, y1, v.centers, v.radii), facecolor="none",
        edgecolor=VALID_COLOR, hatch=VALID_HATCH, linewidth=0, zorder=-4))
    ax.annotate(f"$\\bf{{1}}$\n{outside['range']}", xy=(x0 + 0.10, y0 + 0.10), fontsize=9,
                ha="left", va="bottom", color=text_on(cmap(norm(outside["median_e"]))))
    ax.annotate("design space", xy=(x0 + 0.02, y1 - 0.02), fontsize=8, style="italic",
                va="top", color="0.35")
    ax.set_xlim(x0 - 0.03, x1 + 0.03)
    ax.set_ylim(y0 - 0.03, y1 + 0.03)
    ax.set_aspect("equal")

    sm = ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([-lim, lim])
    ticks = np.linspace(-lim, lim, 7)
    cbar = fig.colorbar(sm, ax=ax, fraction=0.04, pad=0.02, ticks=ticks)
    cbar.set_label(r"median $\frac{\tau_\mathrm{num}-\tau_\mathrm{ana}}{\tau_\mathrm{ana}}$",
                   fontsize=10)
    cbar.ax.set_yticklabels([f"{t * 100:+.0f}%" for t in ticks])
    cbar.ax.tick_params(which="minor", right=False)
    cbar.ax.text(0.5, 1.03, "slower", transform=cbar.ax.transAxes, ha="center", fontsize=7)
    cbar.ax.text(0.5, -0.055, "faster", transform=cbar.ax.transAxes, ha="center", va="top",
                 fontsize=7)

    handles = [plt.Rectangle((0, 0), 1, 1, fc="0.97", ec=VALID_COLOR, lw=0.6,
                             hatch=VALID_HATCH * 2),
               plt.Rectangle((0, 0), 1, 1, fc="0.97", ec=NONEXP_COLOR, lw=0.6,
                             hatch=NONEXP_HATCH * 2)]
    # below the frame, so the legend is not mistaken for part of the design space
    ax.legend(handles, [r"$\tau_\mathrm{ana}$ valid to $\pm5\,\%$",
                        "decay not a single exponential"],
              loc="upper center", bbox_to_anchor=(0.5, -0.01), ncol=2, fontsize=7.5,
              handlelength=1.8, handleheight=1.4)
    return fig


# --------------------------------------------------------------------------- #
# Violin, three normalisations to choose between
# --------------------------------------------------------------------------- #

def _violin_groups(df, value, order):
    data, labels, frac = [], [], []
    for case in order:
        s = df[df.case == case]
        if len(s) == 0:
            continue
        data.append(value(s))
        key = next(k for k, c in CASE_OF.items() if c == case)
        labels.append(f"case {case}\n{REGION_NAME[key]}\n$n={len(s)}$")
        frac.append(s.nonexp.mean())
    return data, labels, frac


def _draw_violins(ax, data, labels, frac, log=False):
    pos = np.arange(len(data))
    parts = ax.violinplot(data, positions=pos, widths=0.82, showextrema=False)
    for body, f in zip(parts["bodies"], frac):
        body.set_facecolor(COLORS["orange"] if f >= NONEXP_HATCH_FRAC else COLORS["grey"])
        body.set_alpha(0.45)
        body.set_edgecolor("0.3")
        body.set_linewidth(0.6)
    ax.boxplot(data, positions=pos, widths=0.15, showfliers=False,
               medianprops=dict(color="0.1", lw=1.2),
               boxprops=dict(color="0.25", lw=0.7),
               whiskerprops=dict(color="0.25", lw=0.7),
               capprops=dict(color="0.25", lw=0.7))
    ax.set_xticks(pos)
    ax.set_xticklabels(labels, fontsize=7)
    categorical_axis(ax, "x")
    ax.grid(False)
    handles = [plt.Rectangle((0, 0), 1, 1, fc=c, alpha=0.45, ec="0.3", lw=0.6)
               for c in (COLORS["grey"], COLORS["orange"])]
    ax.legend(handles, ["exponential decay", "mostly non-exponential"], loc="upper left",
              fontsize=7.5)
    return pos


def make_violin_by_pi(df):
    """C: paired by (Pi, G_P), ranked, with G_mix as the within-pair distinction.

    The pairing is the point: each x position is one (Pi, G_P) combination and holds the two
    G_mix cases side by side. They sit at the same height, which is the finding -- G_mix does
    not move the extraction time. Colour carries G_P, which separates the fast half from the
    slow half without exception, so it stays the dominant visual signal; G_mix is demoted to a
    hatch.
    """
    tau_ref = df.loc[df.case == 1, "tau_fit_nodisc_s"].median()

    # (Pi violated, G_P violated) -> the two cases it contains, ordered G_mix low then high
    pairs = {}
    for key, case in CASE_OF.items():
        pairs.setdefault((key[0], key[2]), {})[key[1]] = case
    order = sorted(pairs, key=lambda g: np.median(
        df.loc[df.case.isin(pairs[g].values()), "tau_fit_nodisc_s"] / tau_ref))

    fig, ax = plt.subplots(figsize=(7.2, 3.8))
    OFFSET = 0.21
    prev_hatch_lw = plt.rcParams["hatch.linewidth"]
    plt.rcParams["hatch.linewidth"] = 0.7      # the G_mix distinction has to be legible
    for i, group in enumerate(order):
        for mix_flag, case in sorted(pairs[group].items()):
            s = df[df.case == case]
            if len(s) == 0:
                continue
            vals = np.log10(s.tau_fit_nodisc_s / tau_ref)
            x = i + (OFFSET if mix_flag else -OFFSET)
            body = ax.violinplot([vals], positions=[x], widths=0.36,
                                 showextrema=False)["bodies"][0]
            body.set_facecolor(COLORS["blue"] if group[1] else COLORS["orange"])
            body.set_alpha(0.40)
            body.set_edgecolor("0.3")
            body.set_linewidth(0.6)
            if mix_flag:
                body.set_hatch("/////")
            # whis=(0,100): whiskers reach the data extremes, so they end exactly where the
            # violin body does. The default 1.5*IQR rule leaves a visible gap that reads as a
            # plotting error.
            ax.boxplot([vals], positions=[x], widths=0.09, showfliers=False, whis=(0, 100),
                       medianprops=dict(color="0.1", lw=1.1),
                       boxprops=dict(color="0.25", lw=0.6),
                       whiskerprops=dict(color="0.25", lw=0.6),
                       capprops=dict(color="0.25", lw=0.6))
            ax.annotate(f"{case}", xy=(x, vals.min()), xytext=(0, -9),
                        textcoords="offset points", ha="center", fontsize=6.5, color="0.4")

    ax.axhline(0, color="0.35", lw=0.9, ls="--", zorder=0)
    ax.set_xticks(np.arange(len(order)))
    ax.set_xticklabels(
        [f"${SYM_PI}{'>' if g[0] else '<'}0.1$\n${SYM_P}{'>' if g[1] else '<'}0.1$"
         for g in order], fontsize=8)
    categorical_axis(ax, "x")
    ax.set_xlim(-0.6, len(order) - 0.4)
    ax.set_ylabel(r"$\log_{10}\left(\tau_\mathrm{num}/\tau_\mathrm{ref}\right)$")
    ax.grid(False)

    sec = ax.secondary_yaxis("right", functions=(lambda v: 10 ** v, np.log10))
    sec.set_yticks([0.03, 0.1, 0.3, 1, 3, 10, 30, 100])
    sec.set_yticklabels([rf"$\times${t:g}" for t in (0.03, 0.1, 0.3, 1, 3, 10, 30, 100)],
                        fontsize=7)
    sec.minorticks_off()
    # rotated 90 deg the string reads bottom to top, so the arrows point the right way
    sec.set_ylabel(r"$\longleftarrow$ faster $\quad$ case 1 $\quad$ slower $\longrightarrow$",
                   fontsize=8)

    fill = [plt.Rectangle((0, 0), 1, 1, fc=COLORS["blue"], alpha=0.40, ec="0.3", lw=0.6),
            plt.Rectangle((0, 0), 1, 1, fc=COLORS["orange"], alpha=0.40, ec="0.3", lw=0.6)]
    hat = [plt.Rectangle((0, 0), 1, 1, fc="0.85", ec="0.3", lw=0.6),
           plt.Rectangle((0, 0), 1, 1, fc="0.85", ec="0.3", lw=0.6, hatch="////")]
    leg1 = ax.legend(fill, [rf"${SYM_P}>0.1$", rf"${SYM_P}<0.1$"], loc="upper left",
                     fontsize=7.5, title="hydrostatic head", title_fontsize=7.5)
    ax.add_artist(leg1)
    ax.legend(hat, [rf"${SYM_MIX}<0.1$", rf"${SYM_MIX}>0.1$"], loc="upper left",
              bbox_to_anchor=(0.22, 1.0), fontsize=7.5, title="liquid mixing",
              title_fontsize=7.5)
    fig.tight_layout()
    plt.rcParams["hatch.linewidth"] = prev_hatch_lw
    return fig


# --------------------------------------------------------------------------- #
# Parity
# --------------------------------------------------------------------------- #

def make_parity(df):
    HOUR = 3600
    # the criterion the study lands on: at most one of the three assumptions violated. That is
    # exactly cases 1, 2, 3 and 5, and it supersedes the two overlapping conditions, which were
    # a subset of it (85 of these 111 samples).
    valid = df.n_violated <= 1
    single = valid & (df.n_violated == 1)
    none_viol = df.n_violated == 0
    rest = ~valid
    x, y = df.tau_pred_s / HOUR, df.tau_fit_nodisc_s / HOUR
    ratio = df.tau_fit_nodisc_s / df.tau_pred_s

    LBL_NONE = "no assumption violated (case 1)"
    LBL_ONE = "exactly one violated (cases 2, 3, 5)"

    fig, (ax, axr) = plt.subplots(1, 2, figsize=(7.4, 3.5))

    # (a) parity. Over four decades a +-5 % band is thinner than the line, so the band lives
    # in panel (b); here the y = x line only shows that nothing is grossly wrong.
    lo, hi = min(x.min(), y.min()) * 0.5, max(x.max(), y.max()) * 2
    line = np.array([lo, hi])
    ax.plot(line, line, color="0.35", lw=0.9, zorder=1)
    ax.scatter(x[rest], y[rest], s=11, c="0.65", alpha=0.35, linewidths=0, zorder=2)
    ax.scatter(x[single], y[single], s=13, c=COLORS["blue"], linewidths=0, zorder=3)
    ax.scatter(x[none_viol], y[none_viol], s=22, facecolors="none",
               edgecolors=COLORS["teal"], linewidths=0.9, zorder=4)
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
    ax.set_aspect("equal")
    ax.set_xlabel(r"$\tau_\mathrm{ana}$ [h]")
    ax.set_ylabel(r"$\tau_\mathrm{num}$ [h]")
    ax.set_title("(a) parity", fontsize=8.5)
    ax.grid(False)

    # (b) the same points as a ratio against the saturation number: this is where the +-5 %
    # claim is legible, and it shows the blue group holding at every Pi
    axr.axhspan(1 - PARITY_BAND, 1 + PARITY_BAND, color="0.5", alpha=0.18, lw=0, zorder=0)
    axr.axhline(1.0, color="0.35", lw=0.9, zorder=1)
    axr.scatter(df.Pi[rest], ratio[rest], s=11, c="0.65", alpha=0.35, linewidths=0, zorder=2)
    axr.scatter(df.Pi[single], ratio[single], s=13, c=COLORS["blue"], linewidths=0, zorder=3)
    axr.scatter(df.Pi[none_viol], ratio[none_viol], s=22, facecolors="none",
                edgecolors=COLORS["teal"], linewidths=0.9, zorder=4)
    axr.axvline(THRESH, color="0.6", lw=0.6, ls=":", zorder=1)
    axr.set_xscale("log")
    axr.set_xlabel(rf"${SYM_PI}$")
    axr.set_ylabel(r"$\tau_\mathrm{num}/\tau_\mathrm{ana}$")
    axr.set_ylim(0.55, 1.25)
    axr.set_title(rf"(b) residual, shaded band $\pm{PARITY_BAND * 100:.0f}\,\%$", fontsize=8.5)
    axr.grid(False)

    handles = [Line2D([], [], ls="none", marker="o", ms=4.2, mfc="none", mec=COLORS["teal"],
                      mew=0.9, label=LBL_NONE),
               Line2D([], [], ls="none", marker="o", ms=3.8, mfc=COLORS["blue"], mec="none",
                      label=LBL_ONE),
               Line2D([], [], ls="none", marker="o", ms=3.5, mfc="0.65", mec="none",
                      label="two or three violated")]
    fig.legend(handles=handles, loc="lower center", ncol=3, fontsize=7, handletextpad=0.4,
               columnspacing=1.2, bbox_to_anchor=(0.5, -0.03))
    fig.tight_layout()

    for name, sel in (("no assumption violated", none_viol),
                      ("exactly one violated", single),
                      ("at most one violated", valid),
                      ("two or more violated", rest)):
        e = df.loc[sel, "e_pred_nodisc"].abs()
        print(f"  parity {name:34s} n={sel.sum():3d}  within +-{PARITY_BAND:.0%}: "
              f"{(e < PARITY_BAND).sum():3d} ({100 * (e < PARITY_BAND).mean():5.1f}%)  "
              f"max|e|={e.max():.3f}")
    return fig


def make_design_map_figures():
    df = load()
    stats = region_stats(df)
    tau_ref = df.loc[df.case == 1, "tau_fit_nodisc_s"].median()
    print(f"tau_ref (median of case 1) = {tau_ref / 3600:.1f} h")
    for case in sorted(CASE_OF.values()):
        s = df[df.case == case]
        key = next(k for k, c in CASE_OF.items() if c == case)
        print(f"  case {case}  n={len(s):3d}  median e={stats[key]['median_e']:+.4f}  "
              f"tau/tau_ref={np.median(s.tau_fit_nodisc_s / tau_ref):7.3f}  "
              f"tau/tau_SPP={np.median(s.tau_fit_nodisc_s / s.tau_pred_SPP_s):6.2f}  "
              f"nonexp={s.nonexp.mean():.0%}")

    FIG_DIR.mkdir(exist_ok=True, parents=True)
    figs = {
        "design_venn": make_venn(stats, df),
        "design_violin": make_violin_by_pi(df),
        "validity_parity_nodisc": make_parity(df),
    }
    for name, fig in figs.items():
        for out in (FIG_DIR / f"{name}.pdf", THESIS_FIG / f"{name}.pdf"):
            fig.savefig(out)
        print(f"wrote {name}.pdf")




In [ ]:
# %% run it
make_design_map_figures()


In [ ]:
# %% ===================== SOBOL SENSITIVITY INDICES =====================
# Sobol sensitivity figure for the LIBRA Pi chapter.
# 
# Dot-and-whisker rather than bars: a Sobol index is a share of variance, not an extensive
# quantity, so filled area would misrepresent it. One panel per transport-property scenario,
# each parameter carrying its first order and total order index side by side, so the reader can
# see the two intervals overlap and conclude that higher order interactions are weak.
# 
# Run from the repository root: python3 make_sobol_figure.py

import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import morethemes as mt

matplotlib.use("Agg")   # headless: this script only writes files
from scipy.stats import sobol_indices

DATA_DIR = Path("data")
FIG_DIR = Path("figures")
THESIS_FIG = Path("figures")

SCENARIOS = ["pessimistic", "optimistic"]          # panel order: pessimistic left
QOI = "t99_nodisc_s"
N_RESAMPLES = 999
BOOTSTRAP_SEED = 0

SCEN_LABEL = {"pessimistic": "pessimistic (Calderoni $K_s$, $D_l$)",
              "optimistic": "optimistic (Malinauskas $K_s$, Fukada $D_l$)"}
PARAM_LABEL = {"temperature": r"$T$", "gas_flow": r"$\dot{n}_g$",
               "top_pressure": r"$P_\mathrm{top}$", "nozzle_diameter": r"$d_\mathrm{noz}$"}

# first order filled, total order open: same hue, so the eye pairs them per parameter
ORDER_STYLE = {
    "first_order": dict(label=r"first order $S_i$", marker="o", mfc="#0077BB", mec="#0077BB"),
    "total_order": dict(label=r"total order $S_{T_i}$", marker="s", mfc="white", mec="#CC3311"),
}
OFFSET = 0.17

mt.set_theme("minimal")
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["STIXGeneral", "Times New Roman", "Times", "DejaVu Serif"],
    "mathtext.fontset": "stix",
    "font.size": 9,
    "axes.labelsize": 9,
    "axes.titlesize": 9,
    "legend.fontsize": 8,
    "xtick.labelsize": 9,
    "ytick.labelsize": 8,
    "axes.linewidth": 0.7,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "ytick.minor.visible": True,
    "legend.frameon": False,
    "savefig.bbox": "tight",
    "savefig.dpi": 200,
})


def sobol_matrices(df, d, n, qoi):
    """Regroup the tagged rows into the {f_A, f_B, f_AB} dict scipy expects.

    The scrambled Sobol prefixes are nested, so the first `n` base samples are a valid
    design on their own. The assertion is the design-completeness check.
    """
    yA, yB = np.full(n, np.nan), np.full(n, np.nan)
    yAB = np.full((d, n), np.nan)
    for role, j, i, val in zip(df.design_role, df.base_index, df.var_index, df[qoi]):
        j = int(j)
        if j >= n:
            continue
        if role == "A":
            yA[j] = val
        elif role == "B":
            yB[j] = val
        else:
            yAB[int(i), j] = val
    assert np.isfinite(yA).all() and np.isfinite(yB).all() and np.isfinite(yAB).all(), (
        f"incomplete Saltelli design for n={n}"
    )
    return {"f_A": yA[None, :], "f_B": yB[None, :], "f_AB": yAB[:, None, :]}


def load(scenario):
    d = DATA_DIR / f"libra_pi_sobol_{scenario}"
    meta = json.load(open(d / "metadata.json"))
    df = pd.read_csv(d / "sobol_data.csv")
    # t99 carries the same backward-Euler bias as tau_fitted; the decay is exponential,
    # so rescaling by tau_nodisc/tau_fitted removes it exactly
    df[QOI] = df.t_extract_99_s * df.tau_fit_nodisc_s / df.tau_fitted_s
    res = sobol_indices(
        func=sobol_matrices(df, meta["d"], meta["n_base_samples"], QOI),
        n=meta["n_base_samples"],
    )
    # scipy's bootstrap exposes no seed, and draws from the global numpy generator, so seeding
    # it here is what makes the confidence intervals reproducible from one run to the next
    np.random.seed(BOOTSTRAP_SEED)
    return dict(
        res=res,
        ci=res.bootstrap(n_resamples=N_RESAMPLES),
        names=[p["name"] for p in meta["param_space"]],
        n=meta["n_base_samples"],
        n_runs=len(df),
    )


def whiskers(value, ci_low, ci_high):
    """Asymmetric error bars, clipped at zero: near-zero indices get NaN bootstrap bounds."""
    err = np.vstack([value - ci_low, ci_high - value])
    return np.nan_to_num(np.clip(err, 0, None))


def make_sobol_figure():
    data = {s: load(s) for s in SCENARIOS}
    names = data[SCENARIOS[0]]["names"]

    # one ordering for both panels, by mean total order, so they stay comparable
    mean_total = np.mean([data[s]["res"].total_order.ravel() for s in SCENARIOS], axis=0)
    order = np.argsort(mean_total)[::-1]
    xpos = np.arange(len(names))

    fig, axes = plt.subplots(1, 2, figsize=(6.8, 2.9), sharey=True)
    for ax, scenario in zip(axes, SCENARIOS):
        d = data[scenario]
        for sign, kind in ((-1, "first_order"), (+1, "total_order")):
            value = getattr(d["res"], kind).ravel()[order]
            interval = getattr(d["ci"], kind).confidence_interval
            err = whiskers(value, interval.low.ravel()[order], interval.high.ravel()[order])
            style = ORDER_STYLE[kind]
            ax.errorbar(
                xpos + sign * OFFSET, value, yerr=err,
                fmt=style["marker"], ms=4.5, mfc=style["mfc"], mec=style["mec"], mew=0.9,
                ecolor=style["mec"], elinewidth=0.9, capsize=2.5, capthick=0.9,
                linestyle="none", label=style["label"], zorder=3,
            )
        # faint separators between parameters, so the pairing is unambiguous
        for x in xpos[:-1]:
            ax.axvline(x + 0.5, color="0.85", lw=0.5, zorder=0)
        ax.axhline(0, color="0.6", lw=0.6, zorder=1)
        ax.set_xticks(xpos)
        ax.set_xticklabels([PARAM_LABEL[names[k]] for k in order])
        # x is categorical and y is a share of variance: neither needs minor ticks
        ax.tick_params(axis="x", which="minor", bottom=False)
        ax.tick_params(axis="y", which="minor", left=False)
        ax.minorticks_off()
        # the theme hides every spine, which leaves the major ticks floating with nothing to
        # sit on; the value axis gets its spine back so the scale is readable
        ax.spines["left"].set_visible(True)
        ax.spines["left"].set_linewidth(0.7)
        ax.set_xlim(-0.6, len(names) - 0.4)
        ax.grid(False)
        ax.set_title(SCEN_LABEL[scenario], fontsize=8.5)

        total_first = getattr(d["res"], "first_order").ravel().sum()
        print(f"{scenario:12s} N={d['n']} ({d['n_runs']} runs)  sum S_i = {total_first:.3f}")
        for k in order:
            s1 = d["res"].first_order.ravel()[k]
            st = d["res"].total_order.ravel()[k]
            print(f"   {names[k]:16s} S_1={s1:6.3f}  S_T={st:6.3f}  S_T-S_1={st - s1:+.3f}")

    axes[0].set_ylabel("Sobol index")
    axes[0].legend(loc="upper right", handletextpad=0.4)
    fig.tight_layout()

    FIG_DIR.mkdir(exist_ok=True, parents=True)
    for out in (FIG_DIR / "sobol_indices.pdf", THESIS_FIG / "sobol_indices.pdf"):
        fig.savefig(out)
        print(f"wrote {out}")




In [ ]:
# %% run it
make_sobol_figure()


In [ ]:
# %% ===================== OPERATING POINT MAP =====================
# Operating-point map for the LIBRA Pi chapter.
# 
# Time to 99 % extraction over the (gas flow, temperature) plane at the design pressure and
# nozzle diameter, one panel per transport-property scenario on a shared colour scale. Read as a
# topographic map: equally spaced isolines whose spacing is the local gradient.
# 
# Same content as CELL B4 of exploration.ipynb, with the colour mesh rasterised so the file stays
# small; isolines, labels and axes remain vector.
# 
# Run from the repository root: python3 make_operating_map.py

import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from matplotlib.ticker import FuncFormatter, NullFormatter
from scipy.interpolate import RBFInterpolator

from sparging.plot_style import apply as apply_style


DATA_DIR = Path("data")
FIG_DIR = Path("figures")
THESIS_FIG = Path("figures")

SCENARIOS = ["pessimistic", "optimistic"]      # panel order: pessimistic left
QOI = "t99_nodisc_s"
HOURS_TO_SEC = 3600

SCEN_LABEL = {"pessimistic": "pessimistic (Calderoni $K_s$, $D_l$)",
              "optimistic": "optimistic (Malinauskas $K_s$, Fukada $D_l$)"}

P_NOM, D_NOM = 1.2, 2.0                 # LIBRA Pi design pressure [atm] and nozzle [mm]
NOMINAL = dict(flow=500, T=550)
N_GRID = 300
RBF_KERNEL, RBF_SMOOTHING = "quintic", 0.0
MAX_ISOLINES = 16
N_SUBDIV = 5                            # intermediate isolines per main interval
LABEL_MARGIN = 0.07                     # keep labels this far (axes fraction) from the edges
RASTER_DPI = 300

apply_style()   # shared style: the theme zeroes the major tick length


def design_matrix(T_C, flow, P_top, d_noz):
    """Surrogate features: log10 on the input whose effect is a power law over a decade."""
    return np.column_stack([np.asarray(T_C, float), np.log10(flow),
                            np.asarray(P_top, float), np.asarray(d_noz, float)])


def fit_tau_surface(df, qoi=QOI):
    """4-D RBF surrogate of log10(qoi in hours). The model is deterministic, so there is no
    noise to average out: a pure interpolant (smoothing = 0) beats a smoothed spline."""
    X = design_matrix(df.temperature_C, df.gas_flow_sccm, df.top_pressure_atm,
                      df.nozzle_diameter_mm)
    mu, sd = X.mean(0), X.std(0)
    rbf = RBFInterpolator((X - mu) / sd, np.log10(df[qoi].to_numpy() / HOURS_TO_SEC),
                          kernel=RBF_KERNEL, smoothing=len(X) * RBF_SMOOTHING)
    return lambda Xn: 10 ** rbf((Xn - mu) / sd)


def contour_interval(vmin, vmax, max_lines=MAX_ISOLINES):
    """First interval of a 1/2/2.5/5/10 ladder that keeps the panel readable."""
    for step in (1, 2, 5, 10, 20, 25, 50, 100, 200, 250, 500, 1000, 2000):
        if (vmax - vmin) / step <= max_lines:
            return step
    return 5000


def inner_label_positions(cs, ax, wanted, margin=LABEL_MARGIN):
    """One label position per wanted level, taken on the contour itself and kept `margin`
    (in axes fraction) away from the four edges, so no label collides with an axis."""
    pos, kept = [], []
    for lev, segs in zip(cs.levels, cs.allsegs):
        if lev not in wanted:
            continue
        best, best_d = None, -np.inf
        for seg in segs:
            if len(seg) < 2:
                continue
            f = ax.transLimits.transform(seg)            # data -> axes fraction
            d = np.minimum(f, 1 - f).min(axis=1)         # distance to nearest edge
            k = int(np.argmax(d))
            if d[k] > best_d:
                best, best_d = seg[k], d[k]
        if best is not None and best_d >= margin:
            pos.append(tuple(best))
            kept.append(lev)
    return pos, kept


def load(scenario):
    d = DATA_DIR / f"libra_pi_sobol_{scenario}"
    df = pd.read_csv(d / "sobol_data.csv")
    df[QOI] = df.t_extract_99_s * df.tau_fit_nodisc_s / df.tau_fitted_s
    return df


def make_operating_map():
    data = {s: load(s) for s in SCENARIOS}

    surf = {}
    for name, df in data.items():
        Tg = np.linspace(df.temperature_C.min(), df.temperature_C.max(), N_GRID)
        Fg = np.linspace(df.gas_flow_sccm.min(), df.gas_flow_sccm.max(), N_GRID)
        TT, FF = np.meshgrid(Tg, Fg, indexing="ij")
        grid = design_matrix(TT.ravel(), FF.ravel(), np.full(TT.size, P_NOM),
                             np.full(TT.size, D_NOM))
        surf[name] = dict(TT=TT, FF=FF, Z=fit_tau_surface(df)(grid).reshape(TT.shape))

    norm = LogNorm(vmin=min(v["Z"].min() for v in surf.values()),
                   vmax=max(v["Z"].max() for v in surf.values()))

    fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.1), sharey=True, constrained_layout=True)
    for ax, name in zip(axes, SCENARIOS):
        v, df = surf[name], data[name]
        # rasterised: a 300x300 gouraud mesh as vectors is ~4 MB on its own
        pcm = ax.pcolormesh(v["FF"], v["TT"], v["Z"], cmap="YlOrRd", norm=norm,
                            shading="gouraud", zorder=0, rasterized=True)
        ax.scatter(df.gas_flow_sccm, df.temperature_C, s=2.6, c="0.15", alpha=0.35,
                   linewidths=0, zorder=1, rasterized=True)
        ax.set_axisbelow(False)
        ax.grid(True, color="0.25", lw=0.3, alpha=0.18, zorder=1.5)

        step = contour_interval(v["Z"].min(), v["Z"].max())
        lo = np.ceil(v["Z"].min() / step) * step
        levels = np.arange(lo, v["Z"].max(), step)
        minor = np.arange(lo - step, v["Z"].max(), step / N_SUBDIV)
        minor = np.array([m for m in minor if m > v["Z"].min()
                          and not np.isclose(m % step, 0, atol=step * 1e-6)])
        ax.contour(v["FF"], v["TT"], v["Z"], levels=minor, colors="0.25", linewidths=0.2,
                   zorder=2)
        cs = ax.contour(v["FF"], v["TT"], v["Z"], levels=levels, colors="0.1",
                        linewidths=0.85, zorder=3)

        # label every main line, but only every second one where they crowd
        crowd = np.median(levels)
        wanted = {lev for i, lev in enumerate(levels) if lev <= crowd or i % 2 == 0}
        pos, kept = inner_label_positions(cs, ax, wanted)
        ax.clabel(cs, manual=pos, fmt=lambda x: f"{x:g}", fontsize=5, inline=True,
                  inline_spacing=1)

        ax.plot(NOMINAL["flow"], NOMINAL["T"], marker="*", ms=13, mfc="white", mec="0.1",
                mew=0.9, zorder=6)
        ax.set_xticks([100, 250, 400, 550, 700, 850, 1000])
        ax.set_xlim(df.gas_flow_sccm.min(), df.gas_flow_sccm.max())
        ax.set_xlabel(r"gas flow rate $\dot{n}_g$ [sccm]")
        ax.set_title(f"{SCEN_LABEL[name]}\ncontour interval {step:g} h "
                     f"({step / N_SUBDIV:g} h intermediate)", fontsize=8)
        print(f"{name:12s} t99 {v['Z'].min():.1f}-{v['Z'].max():.1f} h | interval {step:g} h "
              f"({len(levels)} main + {len(minor)} intermediate) | {len(kept)} labelled "
              f"| {len(df)} design points")

    axes[0].set_ylabel(r"temperature [$^\circ$C]")
    cbar = fig.colorbar(pcm, ax=list(axes), pad=0.02,
                        ticks=[20, 30, 50, 80, 120, 200, 300, 500, 800])
    cbar.ax.yaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{x:g}"))
    cbar.ax.yaxis.set_minor_formatter(NullFormatter())
    cbar.set_label(r"time to 99 % extraction $t_{99}$ [h]")

    FIG_DIR.mkdir(exist_ok=True, parents=True)
    for out in (FIG_DIR / "operating_point_map.pdf", THESIS_FIG / "operating_point_map.pdf"):
        fig.savefig(out, dpi=RASTER_DPI)
        print(f"wrote {out}  ({out.stat().st_size / 1e6:.2f} MB)")




In [ ]:
# %% run it
make_operating_map()


In [ ]:
# NOTE: the thesis now draws this figure in TikZ (masters_thesis/diag_equilibrium.tex),
# generic and without a solubility value. Kept here only in case the plotted version is
# wanted back; delete this cell otherwise.
# %% ===================== CLOSURE RELATIONS: DEPENDENCY GRAPH AND EQUILIBRIUM =========
""" Two small figures for the model chapter.

    The dependency graph moved to make_closure_diagram.py, which emits TikZ so it matches
    the other diagrams of the thesis.
    P(c)  : the interfacial equilibrium of the two-film model, showing the driving force the
            mass transfer coefficient acts on.
"""
# --- P(c) interfacial equilibrium -------------------------------------------------------
inp = get_sim_input_LIBRA_Pi()
K_s = inp.K_s.to("mol/m**3/Pa").magnitude
c_bulk = 3e-11
P_eq = c_bulk / K_s

# the scales are folded into the axis labels: matplotlib's offset text would otherwise sit
# on top of the x label
PS, CS = 1e-8, 1e-11
fig, ax = plt.subplots(figsize=(3.6, 2.9))
P = np.linspace(0, 1.35 * P_eq, 200)
ax.plot(P / PS, K_s * P / CS, color="0.2", lw=1.3, label=r"$c = K_s\,P_{T_2}$ (Henry)")
ax.axhline(c_bulk / CS, color=COLORS["blue"], lw=0.9, ls="--")
ax.plot([P_eq / PS], [c_bulk / CS], marker="o", ms=5, color="0.2", zorder=5)
ax.annotate(r"interface, at equilibrium", xy=(P_eq / PS, c_bulk / CS), xytext=(-8, -16),
            textcoords="offset points", ha="right", fontsize=7)
ax.annotate(r"bulk liquid, $c_{T_2}$", xy=(0.02 * P_eq / PS, c_bulk / CS), xytext=(0, 5),
            textcoords="offset points", fontsize=7, color=COLORS["blue"])
# the driving force the two-film model acts on
P_b = 0.45 * P_eq
ax.annotate("", xy=(P_b / PS, c_bulk / CS), xytext=(P_b / PS, K_s * P_b / CS),
            arrowprops=dict(arrowstyle="<->", color=COLORS["orange"], lw=1.1))
ax.annotate(r"$c_{T_2}-K_s P_{T_2}$", xy=(P_b / PS, 0.5 * (c_bulk + K_s * P_b) / CS), xytext=(6, 0),
            textcoords="offset points", fontsize=7.5, color=COLORS["orange"], va="center")
ax.plot([P_b / PS], [K_s * P_b / CS], marker="o", ms=4, mfc="white", mec="0.2", zorder=5)
ax.annotate("bubble", xy=(P_b / PS, K_s * P_b / CS), xytext=(4, -11), textcoords="offset points",
            fontsize=7)
ax.set_xlabel(r"$P_{T_2}$ in the bubble [$10^{-8}$ Pa]")
ax.set_ylabel(r"$c_{T_2}$ in the liquid [$10^{-11}$ mol m$^{-3}$]")
ax.set_xlim(0, 1.35 * P_eq / PS); ax.set_ylim(0, 1.35 * c_bulk / CS)
ax.legend(loc="upper left", fontsize=7)
ax.grid(False)
fig.tight_layout()
fig.savefig(FIG_DIR / "equilibrium_curve.pdf")
print(f"P(c) curve: K_s = {K_s:.3e} mol/m3/Pa, equilibrium P_T2 = {P_eq:.3e} Pa "
      f"for c = {c_bulk:.1e} mol/m3")


In [ ]:
# %% ===================== PUBLISH : copy only what the thesis uses =====================
""" The notebook produces more figures than the thesis shows (raw as well as discretisation
    corrected variants, single-panel versions kept for inspection). Only the listed files are
    copied into masters_thesis/fig, so that directory stays a faithful picture of the document.
    The audit at the end catches both directions: a figure the thesis wants and we do not
    produce, and a leftover file in the thesis directory that nothing references. """
import re
import shutil

# produced by this notebook and used in thesis.tex
THESIS_FIGURES = [
    "verification_P_T2_profile.pdf",
    "verification_inventory.pdf",
    "convergence_refinement.pdf",
    "validity_error_vs_groups_nodisc.pdf",
    "validity_error_vs_GP_nodisc.pdf",
    "validity_parity_nodisc.pdf",
    "regimes_tau_vs_Pi.pdf",
    "design_tau_vs_Ptop.pdf",
    "nonexp.pdf",
    "stratified.pdf",
    "design_venn.pdf",
    "design_violin.pdf",
    "sobol_indices.pdf",
    "operating_point_map.pdf",
]
# LaTeX fragments belong with the other tables, not in fig/
TABLE_FRAGMENTS = {"convergence_table.tex": Path("../masters_thesis")}

THESIS_FIG_DIR.mkdir(parents=True, exist_ok=True)
changed, identical, missing = [], [], []
for name in THESIS_FIGURES:
    src, dst = FIG_DIR / name, THESIS_FIG_DIR / name
    if not src.exists():
        missing.append(name); continue
    # only overwrite when the bytes differ: with SOURCE_DATE_EPOCH pinned, a figure that was
    # not redrawn is byte-identical, so git shows only the figures that really changed
    if dst.exists() and dst.read_bytes() == src.read_bytes():
        identical.append(name)
    else:
        shutil.copy2(src, dst); changed.append(name)
print(f"published: {len(changed)} changed, {len(identical)} unchanged"
      + (f", {len(missing)} MISSING" if missing else ""))
for n in changed:
    print("   updated:", n)
for n in missing:
    print("   MISSING, not produced:", n)

tex = Path("../masters_thesis")
sources = (tex / "thesis.tex").read_text(encoding="utf-8")
for frag in tex.glob("*.tex"):
    if frag.name != "thesis.tex":
        sources += frag.read_text(encoding="utf-8")
referenced = {m.strip() for m in re.findall(r"\{fig/([^}]+)\}", sources)}
present = {f.name for f in THESIS_FIG_DIR.iterdir() if f.is_file()}

unused = sorted(present - referenced)
missing = sorted(referenced - present)
print(f"thesis references {len(referenced)} figures, {len(present)} present in masters_thesis/fig")
print("  referenced but absent :", missing or "none")
print("  present but unused    :", unused or "none")
if unused:
    print("    (these are not produced here either -- delete by hand or check the reference)")
